In [1]:
# =========================================================
# 📦 IMPORTS + SETUP
# =========================================================
import sys, os, random, time, gc
from pathlib import Path
import yaml
import torch
import pandas as pd

# ---------------------------------------------------------
# 🔥 PROJECT ROOT SETUP
# ---------------------------------------------------------
PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] 
     if (p / "src").exists() and (p / "configs").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Project root not found")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ---------------------------------------------------------
# 🔥 REGISTER CUSTOM MODULES
# ---------------------------------------------------------
import src
import ultralytics.nn.tasks as _tasks
import ultralytics.nn.modules as _modules

from src.custom_modules import M_C3k2, WeightedConcat, HybridSPDConv_3
from src.spd_conv import SPDConv, SPDHybrid, SPDHybrid_NO_Fuse, DKStem

for name, cls in {
    "M_C3k2": M_C3k2,
    "WeightedConcat": WeightedConcat,
    "HybridSPDConv_3": HybridSPDConv_3,
    "SPDConv": SPDConv,
    "SPDHybrid": SPDHybrid,
    "SPDHybrid_NO_Fuse": SPDHybrid_NO_Fuse,
    
    "DKStem": DKStem,
}.items():
    _tasks.__dict__[name] = cls
    _modules.__dict__[name] = cls

from ultralytics import YOLO


# =========================================================
# 🔁 SEED CONTROL
# =========================================================
def set_seed(seed):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


# =========================================================
# 📂 LOAD CONFIG
# =========================================================
def load_config():
    cfg_path = PROJECT_ROOT / "configs" / "base.yaml"
    with open(cfg_path, encoding="utf-8") as f:
        cfg = yaml.safe_load(f)

    cfg["model"] = str((PROJECT_ROOT / cfg["model"]).resolve())
    cfg["experiment"]["project_dir"] = str(
        (PROJECT_ROOT / cfg["experiment"]["project_dir"]).resolve()
    )
    cfg["data"]["config"] = str(
        (PROJECT_ROOT / cfg["data"]["config"]).resolve()
    )

    return cfg


# =========================================================
# 🚀 MAIN PIPELINE
# =========================================================
def run():
    cfg = load_config()

    model_name = cfg["model"]
    training = cfg["training"]
    experiment = cfg["experiment"]
    data_cfg = cfg["data"]["config"]

    seeds = experiment["seeds"]

    all_results = []

    print("\n🚀 Starting Experiments...\n")

    # =====================================
    # 🔁 LOOP OVER SEEDS
    # =====================================
    for seed in seeds:

        print(f"\n========== SEED {seed} ==========\n")

        set_seed(seed)

        exp_name = f"{experiment['name']}_seed{seed}"

        model = YOLO(model_name)

        # --------------------------
        # TRAIN
        # --------------------------
        start = time.time()

        results = model.train(
            data=data_cfg,
            imgsz=training["imgsz"],
            batch=training["batch"],
            epochs=training["epochs"],
            optimizer=training["optimizer"],
            lr0=training["lr0"],
            workers=training["workers"],
            seed=seed,
            project=experiment["project_dir"],
            name=exp_name,
        )

        train_time = (time.time() - start) / 60

        # --------------------------
        # TRAIN METRICS
        # --------------------------
        results_csv = Path(results.save_dir) / "results.csv"
        df = pd.read_csv(results_csv)

        map_col = "metrics/mAP50(B)" if "metrics/mAP50(B)" in df.columns else "metrics/mAP50"
        best_row = df.loc[df[map_col].idxmax()]

        best_weights = Path(results.save_dir) / "weights/best.pt"

        # --------------------------
        # VALIDATION
        # --------------------------
        val_results = model.val(data=data_cfg)

        val_map50 = val_results.box.map50
        val_precision = val_results.box.mp
        val_recall = val_results.box.mr

        speed = val_results.speed
        total_time = sum(speed.values())
        fps = 1000 / total_time if total_time > 0 else 0

        # --------------------------
        # TEST
        # --------------------------
        test_results = model.val(data=data_cfg, split="test")

        test_map50 = test_results.box.map50
        test_precision = test_results.box.mp
        test_recall = test_results.box.mr

        # --------------------------
        # STORE
        # --------------------------
        result = {
            "seed": seed,

            "train_mAP50": best_row.get("metrics/mAP50(B)", best_row.get("metrics/mAP50")),
            "train_precision": best_row.get("metrics/precision(B)", best_row.get("metrics/precision")),
            "train_recall": best_row.get("metrics/recall(B)", best_row.get("metrics/recall")),

            "val_mAP50": val_map50,
            "val_precision": val_precision,
            "val_recall": val_recall,

            "test_mAP50": test_map50,
            "test_precision": test_precision,
            "test_recall": test_recall,

            "fps": fps,
            "train_time_min": train_time,

            "weights": str(best_weights),
        }

        all_results.append(result)

        print("\n📊 Seed Results:")
        for k, v in result.items():
            if k != "weights":
                print(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}")

        del model
        torch.cuda.empty_cache()
        gc.collect()

    # =====================================
    # 📊 FINAL TABLE
    # =====================================
    print("\n📊 ALL SEED RESULTS\n")

    df_results = pd.DataFrame(all_results)
    print(df_results.round(4))

    # =====================================
    # 🏆 BEST MODEL
    # =====================================
    best_exp = df_results.loc[df_results["test_mAP50"].idxmax()]

    print("\n🏆 BEST MODEL (TEST mAP50)\n")
    print(best_exp)

    # =====================================
    # 📈 AVERAGE PERFORMANCE
    # =====================================
    print("\n📈 AVERAGE PERFORMANCE\n")
    print(df_results.mean(numeric_only=True).round(4))

    # =====================================
    # 📊 PER-CLASS METRICS
    # =====================================
    print("\n📊 PER-CLASS PERFORMANCE (TEST SET)\n")

    best_model = YOLO(best_exp["weights"])

    test_results = best_model.val(data=data_cfg, split="test")

    names = test_results.names

    precision_cls = test_results.box.p
    recall_cls = test_results.box.r
    map50_cls = test_results.box.ap50

    class_metrics = []

    for i, name in names.items():
        class_metrics.append({ 
            "class": name,
            "precision": float(precision_cls[i]),
            "recall": float(recall_cls[i]),
            "mAP50": float(map50_cls[i]),
        })

    df_class = pd.DataFrame(class_metrics)
    df_class = df_class.sort_values(by="mAP50", ascending=False)

    print(df_class.round(4))

    print("\n✅ DONE\n")


# =========================================================
# ▶️ RUN
# =========================================================
run()


🚀 Starting Experiments...


========== SEED 1 ==========

Ultralytics 8.4.14  Python-3.11.0 torch-2.1.2+cu118 CUDA:0 (NVIDIA GeForce GTX 1650, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=6, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\MY Projects\Steel Defect Detection\configs\data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=200, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=D:\MY Projects\Steel Defect Detection\models\Baselineyolov8nano.yaml, momentum=

In [1]:
# =========================================================
# 📦 IMPORTS + SETUP
# =========================================================
import sys, os, random, time, gc
from pathlib import Path
import yaml
import torch
import pandas as pd

# ---------------------------------------------------------
# 🔥 PROJECT ROOT SETUP
# ---------------------------------------------------------
PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] 
     if (p / "src").exists() and (p / "configs").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Project root not found")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ---------------------------------------------------------
# 🔥 REGISTER CUSTOM MODULES
# ---------------------------------------------------------
import src
import ultralytics.nn.tasks as _tasks
import ultralytics.nn.modules as _modules

from src.custom_modules import M_C3k2, WeightedConcat, HybridSPDConv_3
from src.spd_conv import SPDConv, SPDHybrid, SPDHybrid_NO_Fuse, DKStem

for name, cls in {
    "M_C3k2": M_C3k2,
    "WeightedConcat": WeightedConcat,
    "HybridSPDConv_3": HybridSPDConv_3,
    "SPDConv": SPDConv,
    "SPDHybrid": SPDHybrid,
    "SPDHybrid_NO_Fuse": SPDHybrid_NO_Fuse,
    
    "DKStem": DKStem,
}.items():
    _tasks.__dict__[name] = cls
    _modules.__dict__[name] = cls

from ultralytics import YOLO


# =========================================================
# 🔁 SEED CONTROL
# =========================================================
def set_seed(seed):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


# =========================================================
# 📂 LOAD CONFIG
# =========================================================
def load_config():
    cfg_path = PROJECT_ROOT / "configs" / "base.yaml"
    with open(cfg_path, encoding="utf-8") as f:
        cfg = yaml.safe_load(f)

    cfg["model"] = str((PROJECT_ROOT / cfg["model"]).resolve())
    cfg["experiment"]["project_dir"] = str(
        (PROJECT_ROOT / cfg["experiment"]["project_dir"]).resolve()
    )
    cfg["data"]["config"] = str(
        (PROJECT_ROOT / cfg["data"]["config"]).resolve()
    )

    return cfg


# =========================================================
# 🚀 MAIN PIPELINE
# =========================================================
def run():
    cfg = load_config()

    model_name = cfg["model"]
    training = cfg["training"]
    experiment = cfg["experiment"]
    data_cfg = cfg["data"]["config"]

    seeds = experiment["seeds"]

    all_results = []

    print("\n🚀 Starting Experiments...\n")

    # =====================================
    # 🔁 LOOP OVER SEEDS
    # =====================================
    for seed in seeds:

        print(f"\n========== SEED {seed} ==========\n")

        set_seed(seed)

        exp_name = f"{experiment['name']}_seed{seed}"

        model = YOLO(model_name)

        # --------------------------
        # TRAIN
        # --------------------------
        start = time.time()

        results = model.train(
            data=data_cfg,
            imgsz=training["imgsz"],
            batch=training["batch"],
            epochs=training["epochs"],
            optimizer=training["optimizer"],
            lr0=training["lr0"],
            workers=training["workers"],
            seed=seed,
            project=experiment["project_dir"],
            name=exp_name,
        )

        train_time = (time.time() - start) / 60

        # --------------------------
        # TRAIN METRICS
        # --------------------------
        results_csv = Path(results.save_dir) / "results.csv"
        df = pd.read_csv(results_csv)

        map_col = "metrics/mAP50(B)" if "metrics/mAP50(B)" in df.columns else "metrics/mAP50"
        best_row = df.loc[df[map_col].idxmax()]

        best_weights = Path(results.save_dir) / "weights/best.pt"

        # --------------------------
        # VALIDATION
        # --------------------------
        val_results = model.val(data=data_cfg)

        val_map50 = val_results.box.map50
        val_precision = val_results.box.mp
        val_recall = val_results.box.mr

        speed = val_results.speed
        total_time = sum(speed.values())
        fps = 1000 / total_time if total_time > 0 else 0

        # --------------------------
        # TEST
        # --------------------------
        test_results = model.val(data=data_cfg, split="test")

        test_map50 = test_results.box.map50
        test_precision = test_results.box.mp
        test_recall = test_results.box.mr

        # --------------------------
        # STORE
        # --------------------------
        result = {
            "seed": seed,

            "train_mAP50": best_row.get("metrics/mAP50(B)", best_row.get("metrics/mAP50")),
            "train_precision": best_row.get("metrics/precision(B)", best_row.get("metrics/precision")),
            "train_recall": best_row.get("metrics/recall(B)", best_row.get("metrics/recall")),

            "val_mAP50": val_map50,
            "val_precision": val_precision,
            "val_recall": val_recall,

            "test_mAP50": test_map50,
            "test_precision": test_precision,
            "test_recall": test_recall,

            "fps": fps,
            "train_time_min": train_time,

            "weights": str(best_weights),
        }

        all_results.append(result)

        print("\n📊 Seed Results:")
        for k, v in result.items():
            if k != "weights":
                print(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}")

        del model
        torch.cuda.empty_cache()
        gc.collect()

    # =====================================
    # 📊 FINAL TABLE
    # =====================================
    print("\n📊 ALL SEED RESULTS\n")

    df_results = pd.DataFrame(all_results)
    print(df_results.round(4))

    # =====================================
    # 🏆 BEST MODEL
    # =====================================
    best_exp = df_results.loc[df_results["test_mAP50"].idxmax()]

    print("\n🏆 BEST MODEL (TEST mAP50)\n")
    print(best_exp)

    # =====================================
    # 📈 AVERAGE PERFORMANCE
    # =====================================
    print("\n📈 AVERAGE PERFORMANCE\n")
    print(df_results.mean(numeric_only=True).round(4))

    # =====================================
    # 📊 PER-CLASS METRICS
    # =====================================
    print("\n📊 PER-CLASS PERFORMANCE (TEST SET)\n")

    best_model = YOLO(best_exp["weights"])

    test_results = best_model.val(data=data_cfg, split="test")

    names = test_results.names

    precision_cls = test_results.box.p
    recall_cls = test_results.box.r
    map50_cls = test_results.box.ap50

    class_metrics = []

    for i, name in names.items():
        class_metrics.append({ 
            "class": name,
            "precision": float(precision_cls[i]),
            "recall": float(recall_cls[i]),
            "mAP50": float(map50_cls[i]),
        })

    df_class = pd.DataFrame(class_metrics)
    df_class = df_class.sort_values(by="mAP50", ascending=False)

    print(df_class.round(4))

    print("\n✅ DONE\n")


# =========================================================
# ▶️ RUN
# =========================================================
run()


🚀 Starting Experiments...


========== SEED 1 ==========

New https://pypi.org/project/ultralytics/8.4.113 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.14  Python-3.11.0 torch-2.1.2+cu118 CUDA:0 (NVIDIA GeForce GTX 1650, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=6, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\MY Projects\Steel Defect Detection\configs\data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=200, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0

In [2]:
# =========================================================
# 📦 IMPORTS + SETUP
# =========================================================
import sys, os, random, time, gc
from pathlib import Path
import yaml
import torch
import pandas as pd

# ---------------------------------------------------------
# 🔥 PROJECT ROOT SETUP
# ---------------------------------------------------------
PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] 
     if (p / "src").exists() and (p / "configs").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Project root not found")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ---------------------------------------------------------
# 🔥 REGISTER CUSTOM MODULES
# ---------------------------------------------------------
import src
import ultralytics.nn.tasks as _tasks
import ultralytics.nn.modules as _modules

from src.custom_modules import M_C3k2, WeightedConcat, HybridSPDConv_3
from src.spd_conv import SPDConv, SPDHybrid, SPDHybrid_NO_Fuse, DKStem

for name, cls in {
    "M_C3k2": M_C3k2,
    "WeightedConcat": WeightedConcat,
    "HybridSPDConv_3": HybridSPDConv_3,
    "SPDConv": SPDConv,
    "SPDHybrid": SPDHybrid,
    "SPDHybrid_NO_Fuse": SPDHybrid_NO_Fuse,
    
    "DKStem": DKStem,
}.items():
    _tasks.__dict__[name] = cls
    _modules.__dict__[name] = cls

from ultralytics import YOLO


# =========================================================
# 🔁 SEED CONTROL
# =========================================================
def set_seed(seed):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


# =========================================================
# 📂 LOAD CONFIG
# =========================================================
def load_config():
    cfg_path = PROJECT_ROOT / "configs" / "base.yaml"
    with open(cfg_path, encoding="utf-8") as f:
        cfg = yaml.safe_load(f)

    cfg["model"] = str((PROJECT_ROOT / cfg["model"]).resolve())
    cfg["experiment"]["project_dir"] = str(
        (PROJECT_ROOT / cfg["experiment"]["project_dir"]).resolve()
    )
    cfg["data"]["config"] = str(
        (PROJECT_ROOT / cfg["data"]["config"]).resolve()
    )

    return cfg


# =========================================================
# 🚀 MAIN PIPELINE
# =========================================================
def run():
    cfg = load_config()

    model_name = cfg["model"]
    training = cfg["training"]
    experiment = cfg["experiment"]
    data_cfg = cfg["data"]["config"]

    seeds = experiment["seeds"]

    all_results = []

    print("\n🚀 Starting Experiments...\n")

    # =====================================
    # 🔁 LOOP OVER SEEDS
    # =====================================
    for seed in seeds:

        print(f"\n========== SEED {seed} ==========\n")

        set_seed(seed)

        exp_name = f"{experiment['name']}_seed{seed}"

        model = YOLO(model_name)

        # --------------------------
        # TRAIN
        # --------------------------
        start = time.time()

        results = model.train(
            data=data_cfg,
            imgsz=training["imgsz"],
            batch=training["batch"],
            epochs=training["epochs"],
            optimizer=training["optimizer"],
            lr0=training["lr0"],
            workers=training["workers"],
            seed=seed,
            project=experiment["project_dir"],
            name=exp_name,
        )

        train_time = (time.time() - start) / 60

        # --------------------------
        # TRAIN METRICS
        # --------------------------
        results_csv = Path(results.save_dir) / "results.csv"
        df = pd.read_csv(results_csv)

        map_col = "metrics/mAP50(B)" if "metrics/mAP50(B)" in df.columns else "metrics/mAP50"
        best_row = df.loc[df[map_col].idxmax()]

        best_weights = Path(results.save_dir) / "weights/best.pt"

        # --------------------------
        # VALIDATION
        # --------------------------
        val_results = model.val(data=data_cfg)

        val_map50 = val_results.box.map50
        val_precision = val_results.box.mp
        val_recall = val_results.box.mr

        speed = val_results.speed
        total_time = sum(speed.values())
        fps = 1000 / total_time if total_time > 0 else 0

        # --------------------------
        # TEST
        # --------------------------
        test_results = model.val(data=data_cfg, split="test")

        test_map50 = test_results.box.map50
        test_precision = test_results.box.mp
        test_recall = test_results.box.mr

        # --------------------------
        # STORE
        # --------------------------
        result = {
            "seed": seed,

            "train_mAP50": best_row.get("metrics/mAP50(B)", best_row.get("metrics/mAP50")),
            "train_precision": best_row.get("metrics/precision(B)", best_row.get("metrics/precision")),
            "train_recall": best_row.get("metrics/recall(B)", best_row.get("metrics/recall")),

            "val_mAP50": val_map50,
            "val_precision": val_precision,
            "val_recall": val_recall,

            "test_mAP50": test_map50,
            "test_precision": test_precision,
            "test_recall": test_recall,

            "fps": fps,
            "train_time_min": train_time,

            "weights": str(best_weights),
        }

        all_results.append(result)

        print("\n📊 Seed Results:")
        for k, v in result.items():
            if k != "weights":
                print(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}")

        del model
        torch.cuda.empty_cache()
        gc.collect()

    # =====================================
    # 📊 FINAL TABLE
    # =====================================
    print("\n📊 ALL SEED RESULTS\n")

    df_results = pd.DataFrame(all_results)
    print(df_results.round(4))

    # =====================================
    # 🏆 BEST MODEL
    # =====================================
    best_exp = df_results.loc[df_results["test_mAP50"].idxmax()]

    print("\n🏆 BEST MODEL (TEST mAP50)\n")
    print(best_exp)

    # =====================================
    # 📈 AVERAGE PERFORMANCE
    # =====================================
    print("\n📈 AVERAGE PERFORMANCE\n")
    print(df_results.mean(numeric_only=True).round(4))

    # =====================================
    # 📊 PER-CLASS METRICS
    # =====================================
    print("\n📊 PER-CLASS PERFORMANCE (TEST SET)\n")

    best_model = YOLO(best_exp["weights"])

    test_results = best_model.val(data=data_cfg, split="test")

    names = test_results.names

    precision_cls = test_results.box.p
    recall_cls = test_results.box.r
    map50_cls = test_results.box.ap50

    class_metrics = []

    for i, name in names.items():
        class_metrics.append({ 
            "class": name,
            "precision": float(precision_cls[i]),
            "recall": float(recall_cls[i]),
            "mAP50": float(map50_cls[i]),
        })

    df_class = pd.DataFrame(class_metrics)
    df_class = df_class.sort_values(by="mAP50", ascending=False)

    print(df_class.round(4))

    print("\n✅ DONE\n")


# =========================================================
# ▶️ RUN
# =========================================================
run()


🚀 Starting Experiments...


========== SEED 1 ==========

WARNING no model scale passed. Assuming scale='n'.
Ultralytics 8.4.14  Python-3.11.0 torch-2.1.2+cu118 CUDA:0 (NVIDIA GeForce GTX 1650, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=6, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\MY Projects\Steel Defect Detection\configs\data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=200, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=D:\MY Projects\Steel Defect 

In [1]:
# =========================================================
# 📦 IMPORTS + SETUP
# =========================================================
import sys, os, random, time, gc
from pathlib import Path
import yaml
import torch
import pandas as pd

# ---------------------------------------------------------
# 🔥 PROJECT ROOT SETUP
# ---------------------------------------------------------
PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] 
     if (p / "src").exists() and (p / "configs").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Project root not found")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ---------------------------------------------------------
# 🔥 REGISTER CUSTOM MODULES
# ---------------------------------------------------------
import src
import ultralytics.nn.tasks as _tasks
import ultralytics.nn.modules as _modules

from src.custom_modules import M_C3k2, WeightedConcat, HybridSPDConv_3
from src.spd_conv import SPDConv, SPDHybrid, SPDHybrid_NO_Fuse, DKStem

for name, cls in {
    "M_C3k2": M_C3k2,
    "WeightedConcat": WeightedConcat,
    "HybridSPDConv_3": HybridSPDConv_3,
    "SPDConv": SPDConv,
    "SPDHybrid": SPDHybrid,
    "SPDHybrid_NO_Fuse": SPDHybrid_NO_Fuse,
    
    "DKStem": DKStem,
}.items():
    _tasks.__dict__[name] = cls
    _modules.__dict__[name] = cls

from ultralytics import YOLO


# =========================================================
# 🔁 SEED CONTROL
# =========================================================
def set_seed(seed):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


# =========================================================
# 📂 LOAD CONFIG
# =========================================================
def load_config():
    cfg_path = PROJECT_ROOT / "configs" / "base.yaml"
    with open(cfg_path, encoding="utf-8") as f:
        cfg = yaml.safe_load(f)

    cfg["model"] = str((PROJECT_ROOT / cfg["model"]).resolve())
    cfg["experiment"]["project_dir"] = str(
        (PROJECT_ROOT / cfg["experiment"]["project_dir"]).resolve()
    )
    cfg["data"]["config"] = str(
        (PROJECT_ROOT / cfg["data"]["config"]).resolve()
    )

    return cfg


# =========================================================
# 🚀 MAIN PIPELINE
# =========================================================
def run():
    cfg = load_config()

    model_name = cfg["model"]
    training = cfg["training"]
    experiment = cfg["experiment"]
    data_cfg = cfg["data"]["config"]

    seeds = experiment["seeds"]

    all_results = []

    print("\n🚀 Starting Experiments...\n")

    # =====================================
    # 🔁 LOOP OVER SEEDS
    # =====================================
    for seed in seeds:

        print(f"\n========== SEED {seed} ==========\n")

        set_seed(seed)

        exp_name = f"{experiment['name']}_seed{seed}"

        model = YOLO(model_name)

        # --------------------------
        # TRAIN
        # --------------------------
        start = time.time()

        results = model.train(
            data=data_cfg,
            imgsz=training["imgsz"],
            batch=training["batch"],
            epochs=training["epochs"],
            optimizer=training["optimizer"],
            lr0=training["lr0"],
            workers=training["workers"],
            seed=seed,
            project=experiment["project_dir"],
            name=exp_name,
        )

        train_time = (time.time() - start) / 60

        # --------------------------
        # TRAIN METRICS
        # --------------------------
        results_csv = Path(results.save_dir) / "results.csv"
        df = pd.read_csv(results_csv)

        map_col = "metrics/mAP50(B)" if "metrics/mAP50(B)" in df.columns else "metrics/mAP50"
        best_row = df.loc[df[map_col].idxmax()]

        best_weights = Path(results.save_dir) / "weights/best.pt"

        # --------------------------
        # VALIDATION
        # --------------------------
        val_results = model.val(data=data_cfg)

        val_map50 = val_results.box.map50
        val_precision = val_results.box.mp
        val_recall = val_results.box.mr

        speed = val_results.speed
        total_time = sum(speed.values())
        fps = 1000 / total_time if total_time > 0 else 0

        # --------------------------
        # TEST
        # --------------------------
        test_results = model.val(data=data_cfg, split="test")

        test_map50 = test_results.box.map50
        test_precision = test_results.box.mp
        test_recall = test_results.box.mr

        # --------------------------
        # STORE
        # --------------------------
        result = {
            "seed": seed,

            "train_mAP50": best_row.get("metrics/mAP50(B)", best_row.get("metrics/mAP50")),
            "train_precision": best_row.get("metrics/precision(B)", best_row.get("metrics/precision")),
            "train_recall": best_row.get("metrics/recall(B)", best_row.get("metrics/recall")),

            "val_mAP50": val_map50,
            "val_precision": val_precision,
            "val_recall": val_recall,

            "test_mAP50": test_map50,
            "test_precision": test_precision,
            "test_recall": test_recall,

            "fps": fps,
            "train_time_min": train_time,

            "weights": str(best_weights),
        }

        all_results.append(result)

        print("\n📊 Seed Results:")
        for k, v in result.items():
            if k != "weights":
                print(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}")

        del model
        torch.cuda.empty_cache()
        gc.collect()

    # =====================================
    # 📊 FINAL TABLE
    # =====================================
    print("\n📊 ALL SEED RESULTS\n")

    df_results = pd.DataFrame(all_results)
    print(df_results.round(4))

    # =====================================
    # 🏆 BEST MODEL
    # =====================================
    best_exp = df_results.loc[df_results["test_mAP50"].idxmax()]

    print("\n🏆 BEST MODEL (TEST mAP50)\n")
    print(best_exp)

    # =====================================
    # 📈 AVERAGE PERFORMANCE
    # =====================================
    print("\n📈 AVERAGE PERFORMANCE\n")
    print(df_results.mean(numeric_only=True).round(4))

    # =====================================
    # 📊 PER-CLASS METRICS
    # =====================================
    print("\n📊 PER-CLASS PERFORMANCE (TEST SET)\n")

    best_model = YOLO(best_exp["weights"])

    test_results = best_model.val(data=data_cfg, split="test")

    names = test_results.names

    precision_cls = test_results.box.p
    recall_cls = test_results.box.r
    map50_cls = test_results.box.ap50

    class_metrics = []

    for i, name in names.items():
        class_metrics.append({ 
            "class": name,
            "precision": float(precision_cls[i]),
            "recall": float(recall_cls[i]),
            "mAP50": float(map50_cls[i]),
        })

    df_class = pd.DataFrame(class_metrics)
    df_class = df_class.sort_values(by="mAP50", ascending=False)

    print(df_class.round(4))

    print("\n✅ DONE\n")


# =========================================================
# ▶️ RUN
# =========================================================
run()


🚀 Starting Experiments...


========== SEED 1 ==========

WARNING no model scale passed. Assuming scale='n'.
Ultralytics 8.4.14  Python-3.11.0 torch-2.1.2+cu118 CUDA:0 (NVIDIA GeForce GTX 1650, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=6, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\MY Projects\Steel Defect Detection\configs\data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=200, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=D:\MY Projects\Steel Defect 

In [2]:
# =========================================================
# 📦 IMPORTS + SETUP
# =========================================================
import sys, os, random, time, gc
from pathlib import Path
import yaml
import torch
import pandas as pd

# ---------------------------------------------------------
# 🔥 PROJECT ROOT SETUP
# ---------------------------------------------------------
PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] 
     if (p / "src").exists() and (p / "configs").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Project root not found")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ---------------------------------------------------------
# 🔥 REGISTER CUSTOM MODULES
# ---------------------------------------------------------
import src
import ultralytics.nn.tasks as _tasks
import ultralytics.nn.modules as _modules

from src.custom_modules import M_C3k2, WeightedConcat, HybridSPDConv_3
from src.spd_conv import SPDConv, SPDHybrid, SPDHybrid_NO_Fuse, DKStem

for name, cls in {
    "M_C3k2": M_C3k2,
    "WeightedConcat": WeightedConcat,
    "HybridSPDConv_3": HybridSPDConv_3,
    "SPDConv": SPDConv,
    "SPDHybrid": SPDHybrid,
    "SPDHybrid_NO_Fuse": SPDHybrid_NO_Fuse,
    
    "DKStem": DKStem,
}.items():
    _tasks.__dict__[name] = cls
    _modules.__dict__[name] = cls

from ultralytics import YOLO


# =========================================================
# 🔁 SEED CONTROL
# =========================================================
def set_seed(seed):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


# =========================================================
# 📂 LOAD CONFIG
# =========================================================
def load_config():
    cfg_path = PROJECT_ROOT / "configs" / "base.yaml"
    with open(cfg_path, encoding="utf-8") as f:
        cfg = yaml.safe_load(f)

    cfg["model"] = str((PROJECT_ROOT / cfg["model"]).resolve())
    cfg["experiment"]["project_dir"] = str(
        (PROJECT_ROOT / cfg["experiment"]["project_dir"]).resolve()
    )
    cfg["data"]["config"] = str(
        (PROJECT_ROOT / cfg["data"]["config"]).resolve()
    )

    return cfg


# =========================================================
# 🚀 MAIN PIPELINE
# =========================================================
def run():
    cfg = load_config()

    model_name = cfg["model"]
    training = cfg["training"]
    experiment = cfg["experiment"]
    data_cfg = cfg["data"]["config"]

    seeds = experiment["seeds"]

    all_results = []

    print("\n🚀 Starting Experiments...\n")

    # =====================================
    # 🔁 LOOP OVER SEEDS
    # =====================================
    for seed in seeds:

        print(f"\n========== SEED {seed} ==========\n")

        set_seed(seed)

        exp_name = f"{experiment['name']}_seed{seed}"

        model = YOLO(model_name)

        # --------------------------
        # TRAIN
        # --------------------------
        start = time.time()

        results = model.train(
            data=data_cfg,
            imgsz=training["imgsz"],
            batch=training["batch"],
            epochs=training["epochs"],
            optimizer=training["optimizer"],
            lr0=training["lr0"],
            workers=training["workers"],
            seed=seed,
            project=experiment["project_dir"],
            name=exp_name,
        )

        train_time = (time.time() - start) / 60

        # --------------------------
        # TRAIN METRICS
        # --------------------------
        results_csv = Path(results.save_dir) / "results.csv"
        df = pd.read_csv(results_csv)

        map_col = "metrics/mAP50(B)" if "metrics/mAP50(B)" in df.columns else "metrics/mAP50"
        best_row = df.loc[df[map_col].idxmax()]

        best_weights = Path(results.save_dir) / "weights/best.pt"

        # --------------------------
        # VALIDATION
        # --------------------------
        val_results = model.val(data=data_cfg)

        val_map50 = val_results.box.map50
        val_precision = val_results.box.mp
        val_recall = val_results.box.mr

        speed = val_results.speed
        total_time = sum(speed.values())
        fps = 1000 / total_time if total_time > 0 else 0

        # --------------------------
        # TEST
        # --------------------------
        test_results = model.val(data=data_cfg, split="test")

        test_map50 = test_results.box.map50
        test_precision = test_results.box.mp
        test_recall = test_results.box.mr

        # --------------------------
        # STORE
        # --------------------------
        result = {
            "seed": seed,

            "train_mAP50": best_row.get("metrics/mAP50(B)", best_row.get("metrics/mAP50")),
            "train_precision": best_row.get("metrics/precision(B)", best_row.get("metrics/precision")),
            "train_recall": best_row.get("metrics/recall(B)", best_row.get("metrics/recall")),

            "val_mAP50": val_map50,
            "val_precision": val_precision,
            "val_recall": val_recall,

            "test_mAP50": test_map50,
            "test_precision": test_precision,
            "test_recall": test_recall,

            "fps": fps,
            "train_time_min": train_time,

            "weights": str(best_weights),
        }

        all_results.append(result)

        print("\n📊 Seed Results:")
        for k, v in result.items():
            if k != "weights":
                print(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}")

        del model
        torch.cuda.empty_cache()
        gc.collect()

    # =====================================
    # 📊 FINAL TABLE
    # =====================================
    print("\n📊 ALL SEED RESULTS\n")

    df_results = pd.DataFrame(all_results)
    print(df_results.round(4))

    # =====================================
    # 🏆 BEST MODEL
    # =====================================
    best_exp = df_results.loc[df_results["test_mAP50"].idxmax()]

    print("\n🏆 BEST MODEL (TEST mAP50)\n")
    print(best_exp)

    # =====================================
    # 📈 AVERAGE PERFORMANCE
    # =====================================
    print("\n📈 AVERAGE PERFORMANCE\n")
    print(df_results.mean(numeric_only=True).round(4))

    # =====================================
    # 📊 PER-CLASS METRICS
    # =====================================
    print("\n📊 PER-CLASS PERFORMANCE (TEST SET)\n")

    best_model = YOLO(best_exp["weights"])

    test_results = best_model.val(data=data_cfg, split="test")

    names = test_results.names

    precision_cls = test_results.box.p
    recall_cls = test_results.box.r
    map50_cls = test_results.box.ap50

    class_metrics = []

    for i, name in names.items():
        class_metrics.append({ 
            "class": name,
            "precision": float(precision_cls[i]),
            "recall": float(recall_cls[i]),
            "mAP50": float(map50_cls[i]),
        })

    df_class = pd.DataFrame(class_metrics)
    df_class = df_class.sort_values(by="mAP50", ascending=False)

    print(df_class.round(4))

    print("\n✅ DONE\n")


# =========================================================
# ▶️ RUN
# =========================================================
run()


🚀 Starting Experiments...


========== SEED 1 ==========

New https://pypi.org/project/ultralytics/8.4.115 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.14  Python-3.11.0 torch-2.1.2+cu118 CUDA:0 (NVIDIA GeForce GTX 1650, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\MY Projects\Steel Defect Detection\configs\data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=300, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0

In [1]:
# =========================================================
# 📦 IMPORTS + SETUP
# =========================================================
import sys, os, random, time, gc
from pathlib import Path
import yaml
import torch
import pandas as pd

# ---------------------------------------------------------
# 🔥 PROJECT ROOT SETUP
# ---------------------------------------------------------
PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] 
     if (p / "src").exists() and (p / "configs").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Project root not found")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ---------------------------------------------------------
# 🔥 REGISTER CUSTOM MODULES
# ---------------------------------------------------------
import src
import ultralytics.nn.tasks as _tasks
import ultralytics.nn.modules as _modules

from src.custom_modules import M_C3k2, WeightedConcat, HybridSPDConv_3
from src.spd_conv import SPDConv, SPDHybrid, SPDHybrid_NO_Fuse, DKStem

for name, cls in {
    "M_C3k2": M_C3k2,
    "WeightedConcat": WeightedConcat,
    "HybridSPDConv_3": HybridSPDConv_3,
    "SPDConv": SPDConv,
    "SPDHybrid": SPDHybrid,
    "SPDHybrid_NO_Fuse": SPDHybrid_NO_Fuse,
    
    "DKStem": DKStem,
}.items():
    _tasks.__dict__[name] = cls
    _modules.__dict__[name] = cls

from ultralytics import YOLO


# =========================================================
# 🔁 SEED CONTROL
# =========================================================
def set_seed(seed):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


# =========================================================
# 📂 LOAD CONFIG
# =========================================================
def load_config():
    cfg_path = PROJECT_ROOT / "configs" / "base.yaml"
    with open(cfg_path, encoding="utf-8") as f:
        cfg = yaml.safe_load(f)

    cfg["model"] = str((PROJECT_ROOT / cfg["model"]).resolve())
    cfg["experiment"]["project_dir"] = str(
        (PROJECT_ROOT / cfg["experiment"]["project_dir"]).resolve()
    )
    cfg["data"]["config"] = str(
        (PROJECT_ROOT / cfg["data"]["config"]).resolve()
    )

    return cfg


# =========================================================
# 🚀 MAIN PIPELINE
# =========================================================
def run():
    cfg = load_config()

    model_name = cfg["model"]
    training = cfg["training"]
    experiment = cfg["experiment"]
    data_cfg = cfg["data"]["config"]

    seeds = experiment["seeds"]

    all_results = []

    print("\n🚀 Starting Experiments...\n")

    # =====================================
    # 🔁 LOOP OVER SEEDS
    # =====================================
    for seed in seeds:

        print(f"\n========== SEED {seed} ==========\n")

        set_seed(seed)

        exp_name = f"{experiment['name']}_seed{seed}"

        model = YOLO(model_name)

        # --------------------------
        # TRAIN
        # --------------------------
        start = time.time()

        results = model.train(
            data=data_cfg,
            imgsz=training["imgsz"],
            batch=training["batch"],
            epochs=training["epochs"],
            optimizer=training["optimizer"],
            lr0=training["lr0"],
            workers=training["workers"],
            seed=seed,
            project=experiment["project_dir"],
            name=exp_name,
        )

        train_time = (time.time() - start) / 60

        # --------------------------
        # TRAIN METRICS
        # --------------------------
        results_csv = Path(results.save_dir) / "results.csv"
        df = pd.read_csv(results_csv)

        map_col = "metrics/mAP50(B)" if "metrics/mAP50(B)" in df.columns else "metrics/mAP50"
        best_row = df.loc[df[map_col].idxmax()]

        best_weights = Path(results.save_dir) / "weights/best.pt"

        # --------------------------
        # VALIDATION
        # --------------------------
        val_results = model.val(data=data_cfg)

        val_map50 = val_results.box.map50
        val_precision = val_results.box.mp
        val_recall = val_results.box.mr

        speed = val_results.speed
        total_time = sum(speed.values())
        fps = 1000 / total_time if total_time > 0 else 0

        # --------------------------
        # TEST
        # --------------------------
        test_results = model.val(data=data_cfg, split="test")

        test_map50 = test_results.box.map50
        test_precision = test_results.box.mp
        test_recall = test_results.box.mr

        # --------------------------
        # STORE
        # --------------------------
        result = {
            "seed": seed,

            "train_mAP50": best_row.get("metrics/mAP50(B)", best_row.get("metrics/mAP50")),
            "train_precision": best_row.get("metrics/precision(B)", best_row.get("metrics/precision")),
            "train_recall": best_row.get("metrics/recall(B)", best_row.get("metrics/recall")),

            "val_mAP50": val_map50,
            "val_precision": val_precision,
            "val_recall": val_recall,

            "test_mAP50": test_map50,
            "test_precision": test_precision,
            "test_recall": test_recall,

            "fps": fps,
            "train_time_min": train_time,

            "weights": str(best_weights),
        }

        all_results.append(result)

        print("\n📊 Seed Results:")
        for k, v in result.items():
            if k != "weights":
                print(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}")

        del model
        torch.cuda.empty_cache()
        gc.collect()

    # =====================================
    # 📊 FINAL TABLE
    # =====================================
    print("\n📊 ALL SEED RESULTS\n")

    df_results = pd.DataFrame(all_results)
    print(df_results.round(4))

    # =====================================
    # 🏆 BEST MODEL
    # =====================================
    best_exp = df_results.loc[df_results["test_mAP50"].idxmax()]

    print("\n🏆 BEST MODEL (TEST mAP50)\n")
    print(best_exp)

    # =====================================
    # 📈 AVERAGE PERFORMANCE
    # =====================================
    print("\n📈 AVERAGE PERFORMANCE\n")
    print(df_results.mean(numeric_only=True).round(4))

    # =====================================
    # 📊 PER-CLASS METRICS
    # =====================================
    print("\n📊 PER-CLASS PERFORMANCE (TEST SET)\n")

    best_model = YOLO(best_exp["weights"])

    test_results = best_model.val(data=data_cfg, split="test")

    names = test_results.names

    precision_cls = test_results.box.p
    recall_cls = test_results.box.r
    map50_cls = test_results.box.ap50

    class_metrics = []

    for i, name in names.items():
        class_metrics.append({ 
            "class": name,
            "precision": float(precision_cls[i]),
            "recall": float(recall_cls[i]),
            "mAP50": float(map50_cls[i]),
        })

    df_class = pd.DataFrame(class_metrics)
    df_class = df_class.sort_values(by="mAP50", ascending=False)

    print(df_class.round(4))

    print("\n✅ DONE\n")


# =========================================================
# ▶️ RUN
# =========================================================
run()


🚀 Starting Experiments...


========== SEED 1 ==========

Ultralytics 8.4.14  Python-3.11.0 torch-2.1.2+cu118 CUDA:0 (NVIDIA GeForce GTX 1650, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\MY Projects\Steel Defect Detection\configs\data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=400, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=D:\MY Projects\Steel Defect Detection\models\Baselineyolov8nano.yaml, momentum=

In [1]:
# =========================================================
# 📦 IMPORTS + SETUP
# =========================================================
import sys, os, random, time, gc
from pathlib import Path
import yaml
import torch
import pandas as pd

# ---------------------------------------------------------
# 🔥 PROJECT ROOT SETUP
# ---------------------------------------------------------
PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] 
     if (p / "src").exists() and (p / "configs").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Project root not found")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ---------------------------------------------------------
# 🔥 REGISTER CUSTOM MODULES
# ---------------------------------------------------------
import src
import ultralytics.nn.tasks as _tasks
import ultralytics.nn.modules as _modules

from src.custom_modules import M_C3k2, WeightedConcat, HybridSPDConv_3
from src.spd_conv import SPDConv, SPDHybrid, SPDHybrid_NO_Fuse, DKStem

for name, cls in {
    "M_C3k2": M_C3k2,
    "WeightedConcat": WeightedConcat,
    "HybridSPDConv_3": HybridSPDConv_3,
    "SPDConv": SPDConv,
    "SPDHybrid": SPDHybrid,
    "SPDHybrid_NO_Fuse": SPDHybrid_NO_Fuse,
    
    "DKStem": DKStem,
}.items():
    _tasks.__dict__[name] = cls
    _modules.__dict__[name] = cls

from ultralytics import YOLO


# =========================================================
# 🔁 SEED CONTROL
# =========================================================
def set_seed(seed):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


# =========================================================
# 📂 LOAD CONFIG
# =========================================================
def load_config():
    cfg_path = PROJECT_ROOT / "configs" / "base.yaml"
    with open(cfg_path, encoding="utf-8") as f:
        cfg = yaml.safe_load(f)

    cfg["model"] = str((PROJECT_ROOT / cfg["model"]).resolve())
    cfg["experiment"]["project_dir"] = str(
        (PROJECT_ROOT / cfg["experiment"]["project_dir"]).resolve()
    )
    cfg["data"]["config"] = str(
        (PROJECT_ROOT / cfg["data"]["config"]).resolve()
    )

    return cfg


# =========================================================
# 🚀 MAIN PIPELINE
# =========================================================
def run():
    cfg = load_config()

    model_name = cfg["model"]
    training = cfg["training"]
    experiment = cfg["experiment"]
    data_cfg = cfg["data"]["config"]

    seeds = experiment["seeds"]

    all_results = []

    print("\n🚀 Starting Experiments...\n")

    # =====================================
    # 🔁 LOOP OVER SEEDS
    # =====================================
    for seed in seeds:

        print(f"\n========== SEED {seed} ==========\n")

        set_seed(seed)

        exp_name = f"{experiment['name']}_seed{seed}"

        model = YOLO(model_name)

        # --------------------------
        # TRAIN
        # --------------------------
        start = time.time()

        results = model.train(
            data=data_cfg,
            imgsz=training["imgsz"],
            batch=training["batch"],
            epochs=training["epochs"],
            optimizer=training["optimizer"],
            lr0=training["lr0"],
            workers=training["workers"],
            seed=seed,
            project=experiment["project_dir"],
            name=exp_name,
        )

        train_time = (time.time() - start) / 60

        # --------------------------
        # TRAIN METRICS
        # --------------------------
        results_csv = Path(results.save_dir) / "results.csv"
        df = pd.read_csv(results_csv)

        map_col = "metrics/mAP50(B)" if "metrics/mAP50(B)" in df.columns else "metrics/mAP50"
        best_row = df.loc[df[map_col].idxmax()]

        best_weights = Path(results.save_dir) / "weights/best.pt"

        # --------------------------
        # VALIDATION
        # --------------------------
        val_results = model.val(data=data_cfg)

        val_map50 = val_results.box.map50
        val_precision = val_results.box.mp
        val_recall = val_results.box.mr

        speed = val_results.speed
        total_time = sum(speed.values())
        fps = 1000 / total_time if total_time > 0 else 0

        # --------------------------
        # TEST
        # --------------------------
        test_results = model.val(data=data_cfg, split="test")

        test_map50 = test_results.box.map50
        test_precision = test_results.box.mp
        test_recall = test_results.box.mr

        # --------------------------
        # STORE
        # --------------------------
        result = {
            "seed": seed,

            "train_mAP50": best_row.get("metrics/mAP50(B)", best_row.get("metrics/mAP50")),
            "train_precision": best_row.get("metrics/precision(B)", best_row.get("metrics/precision")),
            "train_recall": best_row.get("metrics/recall(B)", best_row.get("metrics/recall")),

            "val_mAP50": val_map50,
            "val_precision": val_precision,
            "val_recall": val_recall,

            "test_mAP50": test_map50,
            "test_precision": test_precision,
            "test_recall": test_recall,

            "fps": fps,
            "train_time_min": train_time,

            "weights": str(best_weights),
        }

        all_results.append(result)

        print("\n📊 Seed Results:")
        for k, v in result.items():
            if k != "weights":
                print(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}")

        del model
        torch.cuda.empty_cache()
        gc.collect()

    # =====================================
    # 📊 FINAL TABLE
    # =====================================
    print("\n📊 ALL SEED RESULTS\n")

    df_results = pd.DataFrame(all_results)
    print(df_results.round(4))

    # =====================================
    # 🏆 BEST MODEL
    # =====================================
    best_exp = df_results.loc[df_results["test_mAP50"].idxmax()]

    print("\n🏆 BEST MODEL (TEST mAP50)\n")
    print(best_exp)

    # =====================================
    # 📈 AVERAGE PERFORMANCE
    # =====================================
    print("\n📈 AVERAGE PERFORMANCE\n")
    print(df_results.mean(numeric_only=True).round(4))

    # =====================================
    # 📊 PER-CLASS METRICS
    # =====================================
    print("\n📊 PER-CLASS PERFORMANCE (TEST SET)\n")

    best_model = YOLO(best_exp["weights"])

    test_results = best_model.val(data=data_cfg, split="test")

    names = test_results.names

    precision_cls = test_results.box.p
    recall_cls = test_results.box.r
    map50_cls = test_results.box.ap50

    class_metrics = []

    for i, name in names.items():
        class_metrics.append({ 
            "class": name,
            "precision": float(precision_cls[i]),
            "recall": float(recall_cls[i]),
            "mAP50": float(map50_cls[i]),
        })

    df_class = pd.DataFrame(class_metrics)
    df_class = df_class.sort_values(by="mAP50", ascending=False)

    print(df_class.round(4))

    print("\n✅ DONE\n")


# =========================================================
# ▶️ RUN
# =========================================================
run()


🚀 Starting Experiments...


========== SEED 1 ==========

Ultralytics 8.4.14  Python-3.11.0 torch-2.1.2+cu118 CUDA:0 (NVIDIA GeForce GTX 1650, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\MY Projects\Steel Defect Detection\configs\data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=300, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=D:\MY Projects\Steel Defect Detection\models\V2_DK_SPDHY_yolov8nano.yaml, momen

In [1]:
# =========================================================
# 📦 IMPORTS + SETUP
# =========================================================
import sys, os, random, time, gc
from pathlib import Path
import yaml
import torch
import pandas as pd

# ---------------------------------------------------------
# 🔥 PROJECT ROOT SETUP
# ---------------------------------------------------------
PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] 
     if (p / "src").exists() and (p / "configs").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Project root not found")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ---------------------------------------------------------
# 🔥 REGISTER CUSTOM MODULES
# ---------------------------------------------------------
import src
import ultralytics.nn.tasks as _tasks
import ultralytics.nn.modules as _modules

from src.custom_modules import M_C3k2, WeightedConcat, HybridSPDConv_3
from src.spd_conv import SPDConv, SPDHybrid, SPDHybrid_NO_Fuse, DKStem

for name, cls in {
    "M_C3k2": M_C3k2,
    "WeightedConcat": WeightedConcat,
    "HybridSPDConv_3": HybridSPDConv_3,
    "SPDConv": SPDConv,
    "SPDHybrid": SPDHybrid,
    "SPDHybrid_NO_Fuse": SPDHybrid_NO_Fuse,
    
    "DKStem": DKStem,
}.items():
    _tasks.__dict__[name] = cls
    _modules.__dict__[name] = cls

from ultralytics import YOLO


# =========================================================
# 🔁 SEED CONTROL
# =========================================================
def set_seed(seed):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


# =========================================================
# 📂 LOAD CONFIG
# =========================================================
def load_config():
    cfg_path = PROJECT_ROOT / "configs" / "base.yaml"
    with open(cfg_path, encoding="utf-8") as f:
        cfg = yaml.safe_load(f)

    cfg["model"] = str((PROJECT_ROOT / cfg["model"]).resolve())
    cfg["experiment"]["project_dir"] = str(
        (PROJECT_ROOT / cfg["experiment"]["project_dir"]).resolve()
    )
    cfg["data"]["config"] = str(
        (PROJECT_ROOT / cfg["data"]["config"]).resolve()
    )

    return cfg


# =========================================================
# 🚀 MAIN PIPELINE
# =========================================================
def run():
    cfg = load_config()

    model_name = cfg["model"]
    training = cfg["training"]
    experiment = cfg["experiment"]
    data_cfg = cfg["data"]["config"]

    seeds = experiment["seeds"]

    all_results = []

    print("\n🚀 Starting Experiments...\n")

    # =====================================
    # 🔁 LOOP OVER SEEDS
    # =====================================
    for seed in seeds:

        print(f"\n========== SEED {seed} ==========\n")

        set_seed(seed)

        exp_name = f"{experiment['name']}_seed{seed}"

        model = YOLO(model_name)

        # --------------------------
        # TRAIN
        # --------------------------
        start = time.time()

        results = model.train(
            data=data_cfg,
            imgsz=training["imgsz"],
            batch=training["batch"],
            epochs=training["epochs"],
            optimizer=training["optimizer"],
            lr0=training["lr0"],
            workers=training["workers"],
            seed=seed,
            project=experiment["project_dir"],
            name=exp_name,
        )

        train_time = (time.time() - start) / 60

        # --------------------------
        # TRAIN METRICS
        # --------------------------
        results_csv = Path(results.save_dir) / "results.csv"
        df = pd.read_csv(results_csv)

        map_col = "metrics/mAP50(B)" if "metrics/mAP50(B)" in df.columns else "metrics/mAP50"
        best_row = df.loc[df[map_col].idxmax()]

        best_weights = Path(results.save_dir) / "weights/best.pt"

        # --------------------------
        # VALIDATION
        # --------------------------
        val_results = model.val(data=data_cfg)

        val_map50 = val_results.box.map50
        val_precision = val_results.box.mp
        val_recall = val_results.box.mr

        speed = val_results.speed
        total_time = sum(speed.values())
        fps = 1000 / total_time if total_time > 0 else 0

        # --------------------------
        # TEST
        # --------------------------
        test_results = model.val(data=data_cfg, split="test")

        test_map50 = test_results.box.map50
        test_precision = test_results.box.mp
        test_recall = test_results.box.mr

        # --------------------------
        # STORE
        # --------------------------
        result = {
            "seed": seed,

            "train_mAP50": best_row.get("metrics/mAP50(B)", best_row.get("metrics/mAP50")),
            "train_precision": best_row.get("metrics/precision(B)", best_row.get("metrics/precision")),
            "train_recall": best_row.get("metrics/recall(B)", best_row.get("metrics/recall")),

            "val_mAP50": val_map50,
            "val_precision": val_precision,
            "val_recall": val_recall,

            "test_mAP50": test_map50,
            "test_precision": test_precision,
            "test_recall": test_recall,

            "fps": fps,
            "train_time_min": train_time,

            "weights": str(best_weights),
        }

        all_results.append(result)

        print("\n📊 Seed Results:")
        for k, v in result.items():
            if k != "weights":
                print(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}")

        del model
        torch.cuda.empty_cache()
        gc.collect()

    # =====================================
    # 📊 FINAL TABLE
    # =====================================
    print("\n📊 ALL SEED RESULTS\n")

    df_results = pd.DataFrame(all_results)
    print(df_results.round(4))

    # =====================================
    # 🏆 BEST MODEL
    # =====================================
    best_exp = df_results.loc[df_results["test_mAP50"].idxmax()]

    print("\n🏆 BEST MODEL (TEST mAP50)\n")
    print(best_exp)

    # =====================================
    # 📈 AVERAGE PERFORMANCE
    # =====================================
    print("\n📈 AVERAGE PERFORMANCE\n")
    print(df_results.mean(numeric_only=True).round(4))

    # =====================================
    # 📊 PER-CLASS METRICS
    # =====================================
    print("\n📊 PER-CLASS PERFORMANCE (TEST SET)\n")

    best_model = YOLO(best_exp["weights"])

    test_results = best_model.val(data=data_cfg, split="test")

    names = test_results.names

    precision_cls = test_results.box.p
    recall_cls = test_results.box.r
    map50_cls = test_results.box.ap50

    class_metrics = []

    for i, name in names.items():
        class_metrics.append({ 
            "class": name,
            "precision": float(precision_cls[i]),
            "recall": float(recall_cls[i]),
            "mAP50": float(map50_cls[i]),
        })

    df_class = pd.DataFrame(class_metrics)
    df_class = df_class.sort_values(by="mAP50", ascending=False)

    print(df_class.round(4))

    print("\n✅ DONE\n")


# =========================================================
# ▶️ RUN
# =========================================================
run()


🚀 Starting Experiments...


========== SEED 1 ==========

Ultralytics 8.4.14  Python-3.11.0 torch-2.1.2+cu118 CUDA:0 (NVIDIA GeForce GTX 1650, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\MY Projects\Steel Defect Detection\configs\data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=300, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=D:\MY Projects\Steel Defect Detection\models\V2_DK_SPDHY_yolov8nano.yaml, momen

In [ ]:
# =========================================================
# 📦 IMPORTS + SETUP
# =========================================================
import sys, os, random, time, gc
from pathlib import Path
import yaml
import torch
import pandas as pd

# ---------------------------------------------------------
# 🔥 PROJECT ROOT SETUP
# ---------------------------------------------------------
PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] 
     if (p / "src").exists() and (p / "configs").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Project root not found")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ---------------------------------------------------------
# 🔥 REGISTER CUSTOM MODULES
# ---------------------------------------------------------
import src
import ultralytics.nn.tasks as _tasks
import ultralytics.nn.modules as _modules

from src.custom_modules import M_C3k2, WeightedConcat, HybridSPDConv_3
from src.spd_conv import SPDConv, SPDHybrid, SPDHybrid_NO_Fuse, DKStem

for name, cls in {
    "M_C3k2": M_C3k2,
    "WeightedConcat": WeightedConcat,
    "HybridSPDConv_3": HybridSPDConv_3,
    "SPDConv": SPDConv,
    "SPDHybrid": SPDHybrid,
    "SPDHybrid_NO_Fuse": SPDHybrid_NO_Fuse,
    
    "DKStem": DKStem,
}.items():
    _tasks.__dict__[name] = cls
    _modules.__dict__[name] = cls

from ultralytics import YOLO


# =========================================================
# 🔁 SEED CONTROL
# =========================================================
def set_seed(seed):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


# =========================================================
# 📂 LOAD CONFIG
# =========================================================
def load_config():
    cfg_path = PROJECT_ROOT / "configs" / "base.yaml"
    with open(cfg_path, encoding="utf-8") as f:
        cfg = yaml.safe_load(f)

    cfg["model"] = str((PROJECT_ROOT / cfg["model"]).resolve())
    cfg["experiment"]["project_dir"] = str(
        (PROJECT_ROOT / cfg["experiment"]["project_dir"]).resolve()
    )
    cfg["data"]["config"] = str(
        (PROJECT_ROOT / cfg["data"]["config"]).resolve()
    )

    return cfg


# =========================================================
# 🚀 MAIN PIPELINE
# =========================================================
def run():
    cfg = load_config()

    model_name = cfg["model"]
    training = cfg["training"]
    experiment = cfg["experiment"]
    data_cfg = cfg["data"]["config"]

    seeds = experiment["seeds"]

    all_results = []

    print("\n🚀 Starting Experiments...\n")

    # =====================================
    # 🔁 LOOP OVER SEEDS
    # =====================================
    for seed in seeds:

        print(f"\n========== SEED {seed} ==========\n")

        set_seed(seed)

        exp_name = f"{experiment['name']}_seed{seed}"

        model = YOLO(model_name)

        # --------------------------
        # TRAIN
        # --------------------------
        start = time.time()

        results = model.train(
            data=data_cfg,
            imgsz=training["imgsz"],
            batch=training["batch"],
            epochs=training["epochs"],
            optimizer=training["optimizer"],
            lr0=training["lr0"],
            patience=training["patience"],
            mosaic=training["mosaic"],
            close_mosaic=training["close_mosaic"],
            warmup_epochs=training["warmup_epochs"],
            cos_lr = training["cos_lr"],
            workers=training["workers"],
            seed=seed,
            project=experiment["project_dir"],
            name=exp_name,
        )

        train_time = (time.time() - start) / 60

        # --------------------------
        # TRAIN METRICS
        # --------------------------
        results_csv = Path(results.save_dir) / "results.csv"
        df = pd.read_csv(results_csv)

        map_col = "metrics/mAP50(B)" if "metrics/mAP50(B)" in df.columns else "metrics/mAP50"
        best_row = df.loc[df[map_col].idxmax()]

        best_weights = Path(results.save_dir) / "weights/best.pt"

        # --------------------------
        # VALIDATION
        # --------------------------
        val_results = model.val(data=data_cfg)

        val_map50 = val_results.box.map50
        val_precision = val_results.box.mp
        val_recall = val_results.box.mr

        speed = val_results.speed
        total_time = sum(speed.values())
        fps = 1000 / total_time if total_time > 0 else 0

        # --------------------------
        # TEST
        # --------------------------
        test_results = model.val(data=data_cfg, split="test")

        test_map50 = test_results.box.map50
        test_precision = test_results.box.mp
        test_recall = test_results.box.mr

        # --------------------------
        # STORE
        # --------------------------
        result = {
            "seed": seed,

            "train_mAP50": best_row.get("metrics/mAP50(B)", best_row.get("metrics/mAP50")),
            "train_precision": best_row.get("metrics/precision(B)", best_row.get("metrics/precision")),
            "train_recall": best_row.get("metrics/recall(B)", best_row.get("metrics/recall")),

            "val_mAP50": val_map50,
            "val_precision": val_precision,
            "val_recall": val_recall,

            "test_mAP50": test_map50,
            "test_precision": test_precision,
            "test_recall": test_recall,

            "fps": fps,
            "train_time_min": train_time,
           
            
            "early_stopping": best_row["epoch"] < training["epochs"],
            "weights": str(best_weights),
        }

        all_results.append(result)

        print("\n📊 Seed Results:")
        for k, v in result.items():
            if k != "weights":
                print(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}")



        log_metrics(experiment["name"], result)

        # ======================
        # TEST CSV (separate)
        # ======================
        test_metrics = {
            "seed": seed,
            "mAP50": result["mAP50"],
            "mAP50_95": result["mAP50_95"],
            "precision": result["precision"],
            "recall": result["recall"],
            "fps": fps,
        }

        test_csv_path = Path("results/new exp csv/test_results.csv")

        if test_csv_path.exists():
            test_df = pd.read_csv(test_csv_path)
            test_df = pd.concat([test_df, pd.DataFrame([test_metrics])], ignore_index=True)
        else:
            test_df = pd.DataFrame([test_metrics])

        test_df.to_csv(test_csv_path, index=False)
        



        del model
        torch.cuda.empty_cache()
        gc.collect()

    # =====================================
    # 📊 FINAL TABLE
    # =====================================
    print("\n📊 ALL SEED RESULTS\n")

    df_results = pd.DataFrame(all_results)
    print(df_results.round(4))

    # =====================================
    # 🏆 BEST MODEL
    # =====================================
    best_exp = df_results.loc[df_results["test_mAP50"].idxmax()]

    print("\n🏆 BEST MODEL (TEST mAP50)\n")
    print(best_exp)

    # =====================================
    # 📈 AVERAGE PERFORMANCE
    # =====================================
    print("\n📈 AVERAGE PERFORMANCE\n")
    print(df_results.mean(numeric_only=True).round(4))

    # =====================================
    # 📊 PER-CLASS METRICS
    # =====================================
    print("\n📊 PER-CLASS PERFORMANCE (TEST SET)\n")

    best_model = YOLO(best_exp["weights"])

    test_results = best_model.val(data=data_cfg, split="test")

    names = test_results.names

    precision_cls = test_results.box.p
    recall_cls = test_results.box.r
    map50_cls = test_results.box.ap50

    class_metrics = []

    for i, name in names.items():
        class_metrics.append({ 
            "class": name,
            "precision": float(precision_cls[i]),
            "recall": float(recall_cls[i]),
            "mAP50": float(map50_cls[i]),
        })

    df_class = pd.DataFrame(class_metrics)
    df_class = df_class.sort_values(by="mAP50", ascending=False)

    print(df_class.round(4))

    print("\n✅ DONE\n")


# =========================================================
# ▶️ RUN
# =========================================================
run()


🚀 Starting Experiments...


========== SEED 1 ==========

WARNING no model scale passed. Assuming scale='n'.
Ultralytics 8.4.14  Python-3.11.0 torch-2.1.2+cu118 CUDA:0 (NVIDIA GeForce GTX 1650, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=20, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=D:\MY Projects\Steel Defect Detection\configs\data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=300, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=D:\MY Projects\Steel Defect D

NameError: name 'gpu_memory' is not defined

In [1]:
# =========================================================
# 📦 IMPORTS + SETUP
# =========================================================
import sys, os, random, time, gc
from pathlib import Path
from datetime import datetime
import yaml
import torch
import pandas as pd

# ---------------------------------------------------------
# 🔥 PROJECT ROOT SETUP
# ---------------------------------------------------------
PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "src").exists() and (p / "configs").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Project root not found")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ---------------------------------------------------------
# 🔥 REGISTER CUSTOM MODULES
# ---------------------------------------------------------
import src
import ultralytics.nn.tasks as _tasks
import ultralytics.nn.modules as _modules

from src.custom_modules import M_C3k2, WeightedConcat, HybridSPDConv_3
from src.spd_conv import SPDConv, SPDHybrid, SPDHybrid_NO_Fuse, DKStem

for name, cls in {
    "M_C3k2": M_C3k2,
    "WeightedConcat": WeightedConcat,
    "HybridSPDConv_3": HybridSPDConv_3,
    "SPDConv": SPDConv,
    "SPDHybrid": SPDHybrid,
    "SPDHybrid_NO_Fuse": SPDHybrid_NO_Fuse,
    "DKStem": DKStem,
}.items():
    _tasks.__dict__[name] = cls
    _modules.__dict__[name] = cls

from ultralytics import YOLO


# =========================================================
# 🔁 SEED CONTROL
# =========================================================
def set_seed(seed):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


# =========================================================
# 📂 LOAD CONFIG
# =========================================================
def load_config():
    cfg_path = PROJECT_ROOT / "configs" / "base.yaml"
    with open(cfg_path, encoding="utf-8") as f:
        cfg = yaml.safe_load(f)

    cfg["model"] = str((PROJECT_ROOT / cfg["model"]).resolve())
    cfg["experiment"]["project_dir"] = str(
        (PROJECT_ROOT / cfg["experiment"]["project_dir"]).resolve()
    )
    cfg["data"]["config"] = str(
        (PROJECT_ROOT / cfg["data"]["config"]).resolve()
    )

    return cfg


# =========================================================
# 🧮 MODEL COMPLEXITY (params + GFLOPs ONLY)
# =========================================================
def get_model_info(weights_path, imgsz=640):
    """
    Loads a checkpoint fresh and calculates total params and GFLOPs directly.
    Returns zero for layers/gradients so downstream table logging doesn't break.
    """
    info_model = YOLO(str(weights_path))
    model = info_model.model

    # 1. Direct PyTorch Parameter Count
    n_p = sum(p.numel() for p in model.parameters())

    # 2. Extract GFLOPs safely
    flops = 0.0
    try:
        info_out = model.info(detailed=False, verbose=False, imgsz=imgsz)
        if isinstance(info_out, (tuple, list)) and len(info_out) >= 4:
            flops = info_out[3]
        elif hasattr(model, "flops"):
            flops = model.flops
    except Exception as e:
        print(f"[warning] Could not calculate GFLOPs: {e}")

    del info_model
    return {"params": n_p, "gflops": flops, "layers": 0, "gradients": 0}


# =========================================================
# 🚨 OVERFITTING / VAL-LOSS-DIVERGENCE MONITOR
# =========================================================
def make_overfit_monitor(window=5, exp_name=""):
    """
    Registers as an `on_fit_epoch_end` callback. Watches for the classic
    overfitting signature: train loss still falling while val loss climbs,
    measured over a rolling `window` of epochs.
    """
    history = {"epoch": [], "train_loss": [], "val_loss": []}

    def _callback(trainer):
        try:
            m = trainer.metrics or {}
            val_loss = sum(
                m.get(k, 0.0) for k in ("val/box_loss", "val/cls_loss", "val/dfl_loss")
            )
            train_loss = (
                float(trainer.tloss.sum())
                if getattr(trainer, "tloss", None) is not None
                else None
            )

            if train_loss is None or val_loss == 0.0:
                return  # metrics not populated yet this epoch

            history["epoch"].append(trainer.epoch)
            history["train_loss"].append(train_loss)
            history["val_loss"].append(val_loss)

            if len(history["epoch"]) >= window:
                t_now, t_prev = history["train_loss"][-1], history["train_loss"][-window]
                v_now, v_prev = history["val_loss"][-1], history["val_loss"][-window]

                if (t_now < t_prev) and (v_now > v_prev):
                    print(
                        f"\n🚨 [{exp_name}] Epoch {trainer.epoch}: "
                        f"train loss ↓ ({t_prev:.4f} → {t_now:.4f}) but "
                        f"val loss ↑ ({v_prev:.4f} → {v_now:.4f}) over last {window} epochs.\n"
                        f"    Looks like overfitting starting — consider Kernel → Interrupt "
                        f"if this keeps repeating.\n"
                    )
        except Exception as e:
            print(f"[monitor warning] could not evaluate loss trend: {e}")

    return _callback


# =========================================================
# 📝 MASTER CSV LOGGER
# =========================================================
def log_experiment_to_csv(cfg, seed, exp_name, result, model_info, csv_path):
    """
    Appends one full row per seed/run to a master CSV.
    """
    training = cfg["training"]

    row = {
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "experiment_name": exp_name,
        "model_config": cfg["model"],
        "dataset_config": cfg["data"]["config"],
        "seed": seed,

        # --- Model complexity ---
        "params_M": round(model_info["params"] / 1e6, 4),
        "gflops": round(model_info["gflops"], 4),
        "layers": model_info["layers"],

        # --- Training setup ---
        "epochs_planned": training["epochs"],
        "best_epoch": result["best_epoch"],
        "early_stopped": result["early_stopping"],
        "optimizer": training["optimizer"],
        "lr0": training["lr0"],
        "batch": training["batch"],
        "imgsz": training["imgsz"],
        "save_period": training.get("save_period", 50),

        # --- Train (best epoch) metrics ---
        "train_mAP50": result["train_mAP50"],
        "train_precision": result["train_precision"],
        "train_recall": result["train_recall"],

        # --- Val metrics ---
        "val_mAP50": result["val_mAP50"],
        "val_mAP50_95": result["val_mAP50_95"],
        "val_precision": result["val_precision"],
        "val_recall": result["val_recall"],

        # --- Test metrics ---
        "test_mAP50": result["test_mAP50"],
        "test_mAP50_95": result["test_mAP50_95"],
        "test_precision": result["test_precision"],
        "test_recall": result["test_recall"],

        # --- Efficiency ---
        "fps": result["fps"],
        "train_time_min": result["train_time_min"],

        "weights_path": result["weights"],
    }

    csv_path = Path(csv_path)
    csv_path.parent.mkdir(parents=True, exist_ok=True)

    df_row = pd.DataFrame([row])
    if csv_path.exists():
        df_row.to_csv(csv_path, mode="a", header=False, index=False)
    else:
        df_row.to_csv(csv_path, mode="w", header=True, index=False)

    return row


# =========================================================
# 🚀 MAIN PIPELINE
# =========================================================
def run():
    cfg = load_config()

    model_name = cfg["model"]
    training = cfg["training"]
    experiment = cfg["experiment"]
    data_cfg = cfg["data"]["config"]

    seeds = experiment["seeds"]

    master_csv_path = experiment.get(
        "log_csv", PROJECT_ROOT / "results" / "experiment_log.csv"
    )

    all_results = []

    print("\n🚀 Starting Experiments...\n")

    # =====================================
    # 🔁 LOOP OVER SEEDS
    # =====================================
    for seed in seeds:

        print(f"\n========== SEED {seed} ==========\n")

        set_seed(seed)

        exp_name = f"{experiment['name']}_seed{seed}"
        exp_dir = Path(experiment["project_dir"]) / exp_name
        last_ckpt = exp_dir / "weights" / "last.pt"

        # --------------------------
        # ✅ RESUME CHECKPOINT
        # --------------------------
        if last_ckpt.exists():
            print(f"⚠️  Found existing checkpoint for {exp_name} — resuming from {last_ckpt}")
            model = YOLO(str(last_ckpt))
            train_kwargs = dict(resume=True)
        else:
            model = YOLO(model_name)
            train_kwargs = dict(
                data=data_cfg,
                imgsz=training["imgsz"],
                batch=training["batch"],
                epochs=training["epochs"],
                optimizer=training["optimizer"],
                lr0=training["lr0"],
                patience=training["patience"],
                mosaic=training["mosaic"],
                close_mosaic=training["close_mosaic"],
                warmup_epochs=training["warmup_epochs"],
                cos_lr=training["cos_lr"],
                workers=training["workers"],
                seed=seed,
                project=experiment["project_dir"],
                name=exp_name,
                save_period=training.get("save_period", 50),
            )

        # --------------------------
        # ✅ VAL-LOSS MONITOR
        # --------------------------
        model.add_callback(
            "on_fit_epoch_end",
            make_overfit_monitor(window=training.get("overfit_window", 30), exp_name=exp_name),
        )

        # --------------------------
        # TRAIN
        # --------------------------
        start = time.time()

        try:
            results = model.train(**train_kwargs)
        except KeyboardInterrupt:
            print(f"\n⏸️  Training manually interrupted for {exp_name}.")
            print(f"    Last checkpoint should be safe at: {last_ckpt}")
            print("    Re-run this cell — it will auto-resume from that checkpoint.")
            break

        train_time = (time.time() - start) / 60

        # --------------------------
        # TRAIN METRICS
        # --------------------------
        results_csv = Path(results.save_dir) / "results.csv"
        df = pd.read_csv(results_csv)

        map_col = "metrics/mAP50(B)" if "metrics/mAP50(B)" in df.columns else "metrics/mAP50"
        best_row = df.loc[df[map_col].idxmax()]

        best_weights = Path(results.save_dir) / "weights/best.pt"

        # --------------------------
        # VALIDATION
        # --------------------------
        val_results = model.val(data=data_cfg)

        val_map50 = val_results.box.map50
        val_map50_95 = val_results.box.map
        val_precision = val_results.box.mp
        val_recall = val_results.box.mr

        speed = val_results.speed
        total_time = sum(speed.values())
        fps = 1000 / total_time if total_time > 0 else 0

        # --------------------------
        # TEST
        # --------------------------
        test_results = model.val(data=data_cfg, split="test")

        test_map50 = test_results.box.map50
        test_map50_95 = test_results.box.map
        test_precision = test_results.box.mp
        test_recall = test_results.box.mr

        # --------------------------
        # MODEL COMPLEXITY (params + GFLOPs)
        # --------------------------
        model_info = get_model_info(best_weights, imgsz=training["imgsz"])

        # --------------------------
        # STORE
        # --------------------------
        result = {
            "seed": seed,
            "best_epoch": int(best_row["epoch"]),

            "train_mAP50": best_row.get("metrics/mAP50(B)", best_row.get("metrics/mAP50")),
            "train_precision": best_row.get("metrics/precision(B)", best_row.get("metrics/precision")),
            "train_recall": best_row.get("metrics/recall(B)", best_row.get("metrics/recall")),

            "val_mAP50": val_map50,
            "val_mAP50_95": val_map50_95,
            "val_precision": val_precision,
            "val_recall": val_recall,

            "test_mAP50": test_map50,
            "test_mAP50_95": test_map50_95,
            "test_precision": test_precision,
            "test_recall": test_recall,

            "fps": fps,
            "train_time_min": train_time,

            "early_stopping": bool(best_row["epoch"] < training["epochs"]),
            "weights": str(best_weights),
        }

        all_results.append(result)

        print("\n📊 Seed Results:")
        for k, v in result.items():
            if k != "weights":
                print(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}")

        # --------------------------
        # ✅ CSV LOG
        # --------------------------
        log_experiment_to_csv(cfg, seed, exp_name, result, model_info, master_csv_path)
        print(f"📝 Logged to {master_csv_path}")

        del model
        torch.cuda.empty_cache()
        gc.collect()

    # =====================================
    # 📊 FINAL TABLE
    # =====================================
    if not all_results:
        print("\n⚠️  No completed runs to summarize (training interrupted before any seed finished).\n")
        return

    print("\n📊 ALL SEED RESULTS\n")

    df_results = pd.DataFrame(all_results)
    print(df_results.round(4))

    # =====================================
    # 🏆 BEST MODEL
    # =====================================
    best_exp = df_results.loc[df_results["test_mAP50"].idxmax()]

    print("\n🏆 BEST MODEL (TEST mAP50)\n")
    print(best_exp)

    # =====================================
    # 📈 AVERAGE PERFORMANCE
    # =====================================
    print("\n📈 AVERAGE PERFORMANCE\n")
    print(df_results.mean(numeric_only=True).round(4))

    # =====================================
    # 📊 PER-CLASS METRICS
    # =====================================
    print("\n📊 PER-CLASS PERFORMANCE (TEST SET)\n")

    best_model = YOLO(best_exp["weights"])

    test_results = best_model.val(data=data_cfg, split="test")

    names = test_results.names

    precision_cls = test_results.box.p
    recall_cls = test_results.box.r
    map50_cls = test_results.box.ap50

    class_metrics = []

    for i, name in names.items():
        class_metrics.append({
            "class": name,
            "precision": float(precision_cls[i]),
            "recall": float(recall_cls[i]),
            "mAP50": float(map50_cls[i]),
        })

    df_class = pd.DataFrame(class_metrics)
    df_class = df_class.sort_values(by="mAP50", ascending=False)

    print(df_class.round(4))

    print(f"\n✅ DONE — full experiment log at: {master_csv_path}\n")


# =========================================================
# ▶️ RUN
# =========================================================
run()


🚀 Starting Experiments...


========== SEED 1 ==========

⚠️  Found existing checkpoint for NEUDET__YOLO_v11n_SGD__300epochs_seed1 — resuming from D:\MY Projects\Steel Defect Detection\runs\detect\experiments\NEUDET__YOLO_v11n_SGD__300epochs_seed1\weights\last.pt
Ultralytics 8.4.14  Python-3.11.0 torch-2.1.2+cu118 CUDA:0 (NVIDIA GeForce GTX 1650, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=20, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\MY Projects\Steel Defect Detection\configs\data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=300, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False,

In [1]:
# =========================================================
# 📦 IMPORTS + SETUP
# =========================================================
import sys, os, random, time, gc
from pathlib import Path
from datetime import datetime
import yaml
import torch
import pandas as pd

# ---------------------------------------------------------
# 🔥 PROJECT ROOT SETUP
# ---------------------------------------------------------
PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "src").exists() and (p / "configs").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Project root not found")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ---------------------------------------------------------
# 🔥 REGISTER CUSTOM MODULES
# ---------------------------------------------------------
import src
import ultralytics.nn.tasks as _tasks
import ultralytics.nn.modules as _modules

from src.custom_modules import M_C3k2, WeightedConcat, HybridSPDConv_3
from src.spd_conv import SPDConv, SPDHybrid, SPDHybrid_NO_Fuse, DKStem

for name, cls in {
    "M_C3k2": M_C3k2,
    "WeightedConcat": WeightedConcat,
    "HybridSPDConv_3": HybridSPDConv_3,
    "SPDConv": SPDConv,
    "SPDHybrid": SPDHybrid,
    "SPDHybrid_NO_Fuse": SPDHybrid_NO_Fuse,
    "DKStem": DKStem,
}.items():
    _tasks.__dict__[name] = cls
    _modules.__dict__[name] = cls

from ultralytics import YOLO


# =========================================================
# 🔁 SEED CONTROL
# =========================================================
def set_seed(seed):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


# =========================================================
# 📂 LOAD CONFIG
# =========================================================
def load_config():
    cfg_path = PROJECT_ROOT / "configs" / "base.yaml"
    with open(cfg_path, encoding="utf-8") as f:
        cfg = yaml.safe_load(f)

    cfg["model"] = str((PROJECT_ROOT / cfg["model"]).resolve())
    cfg["experiment"]["project_dir"] = str(
        (PROJECT_ROOT / cfg["experiment"]["project_dir"]).resolve()
    )
    cfg["data"]["config"] = str(
        (PROJECT_ROOT / cfg["data"]["config"]).resolve()
    )

    return cfg


# =========================================================
# 🧮 MODEL COMPLEXITY (params + GFLOPs ONLY)
# =========================================================
def get_model_info(weights_path, imgsz=640):
    """
    Loads a checkpoint fresh and calculates total params and GFLOPs directly.
    Returns zero for layers/gradients so downstream table logging doesn't break.
    """
    info_model = YOLO(str(weights_path))
    model = info_model.model

    # 1. Direct PyTorch Parameter Count
    n_p = sum(p.numel() for p in model.parameters())

    # 2. Extract GFLOPs safely
    flops = 0.0
    try:
        info_out = model.info(detailed=False, verbose=False, imgsz=imgsz)
        if isinstance(info_out, (tuple, list)) and len(info_out) >= 4:
            flops = info_out[3]
        elif hasattr(model, "flops"):
            flops = model.flops
    except Exception as e:
        print(f"[warning] Could not calculate GFLOPs: {e}")

    del info_model
    return {"params": n_p, "gflops": flops, "layers": 0, "gradients": 0}


# =========================================================
# 🚨 OVERFITTING / VAL-LOSS-DIVERGENCE MONITOR
# =========================================================
def make_overfit_monitor(window=5, exp_name=""):
    """
    Registers as an `on_fit_epoch_end` callback. Watches for the classic
    overfitting signature: train loss still falling while val loss climbs,
    measured over a rolling `window` of epochs.
    """
    history = {"epoch": [], "train_loss": [], "val_loss": []}

    def _callback(trainer):
        try:
            m = trainer.metrics or {}
            val_loss = sum(
                m.get(k, 0.0) for k in ("val/box_loss", "val/cls_loss", "val/dfl_loss")
            )
            train_loss = (
                float(trainer.tloss.sum())
                if getattr(trainer, "tloss", None) is not None
                else None
            )

            if train_loss is None or val_loss == 0.0:
                return  # metrics not populated yet this epoch

            history["epoch"].append(trainer.epoch)
            history["train_loss"].append(train_loss)
            history["val_loss"].append(val_loss)

            if len(history["epoch"]) >= window:
                t_now, t_prev = history["train_loss"][-1], history["train_loss"][-window]
                v_now, v_prev = history["val_loss"][-1], history["val_loss"][-window]

                if (t_now < t_prev) and (v_now > v_prev):
                    print(
                        f"\n🚨 [{exp_name}] Epoch {trainer.epoch}: "
                        f"train loss ↓ ({t_prev:.4f} → {t_now:.4f}) but "
                        f"val loss ↑ ({v_prev:.4f} → {v_now:.4f}) over last {window} epochs.\n"
                        f"    Looks like overfitting starting — consider Kernel → Interrupt "
                        f"if this keeps repeating.\n"
                    )
        except Exception as e:
            print(f"[monitor warning] could not evaluate loss trend: {e}")

    return _callback


# =========================================================
# 📝 MASTER CSV LOGGER
# =========================================================
def log_experiment_to_csv(cfg, seed, exp_name, result, model_info, csv_path):
    """
    Appends one full row per seed/run to a master CSV.
    """
    training = cfg["training"]

    row = {
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "experiment_name": exp_name,
        "model_config": cfg["model"],
        "dataset_config": cfg["data"]["config"],
        "seed": seed,

        # --- Model complexity ---
        "params_M": round(model_info["params"] / 1e6, 4),
        "gflops": round(model_info["gflops"], 4),
        "layers": model_info["layers"],

        # --- Training setup ---
        "epochs_planned": training["epochs"],
        "best_epoch": result["best_epoch"],
        "early_stopped": result["early_stopping"],
        "optimizer": training["optimizer"],
        "lr0": training["lr0"],
        "batch": training["batch"],
        "imgsz": training["imgsz"],
        "save_period": training.get("save_period", 50),

        # --- Train (best epoch) metrics ---
        "train_mAP50": result["train_mAP50"],
        "train_precision": result["train_precision"],
        "train_recall": result["train_recall"],

        # --- Val metrics ---
        "val_mAP50": result["val_mAP50"],
        "val_mAP50_95": result["val_mAP50_95"],
        "val_precision": result["val_precision"],
        "val_recall": result["val_recall"],

        # --- Test metrics ---
        "test_mAP50": result["test_mAP50"],
        "test_mAP50_95": result["test_mAP50_95"],
        "test_precision": result["test_precision"],
        "test_recall": result["test_recall"],

        # --- Efficiency ---
        "fps": result["fps"],
        "train_time_min": result["train_time_min"],

        "weights_path": result["weights"],
    }

    csv_path = Path(csv_path)
    csv_path.parent.mkdir(parents=True, exist_ok=True)

    df_row = pd.DataFrame([row])
    if csv_path.exists():
        df_row.to_csv(csv_path, mode="a", header=False, index=False)
    else:
        df_row.to_csv(csv_path, mode="w", header=True, index=False)

    return row


# =========================================================
# 🚀 MAIN PIPELINE
# =========================================================
def run():
    cfg = load_config()

    model_name = cfg["model"]
    training = cfg["training"]
    experiment = cfg["experiment"]
    data_cfg = cfg["data"]["config"]

    seeds = experiment["seeds"]

    master_csv_path = experiment.get(
        "log_csv", PROJECT_ROOT / "results" / "experiment_log.csv"
    )

    all_results = []

    print("\n🚀 Starting Experiments...\n")

    # =====================================
    # 🔁 LOOP OVER SEEDS
    # =====================================
    for seed in seeds:

        print(f"\n========== SEED {seed} ==========\n")

        set_seed(seed)

        exp_name = f"{experiment['name']}_seed{seed}"
        exp_dir = Path(experiment["project_dir"]) / exp_name
        last_ckpt = exp_dir / "weights" / "last.pt"

        # --------------------------
        # ✅ RESUME CHECKPOINT
        # --------------------------
        if last_ckpt.exists():
            print(f"⚠️  Found existing checkpoint for {exp_name} — resuming from {last_ckpt}")
            model = YOLO(str(last_ckpt))
            train_kwargs = dict(resume=True)
        else:
            model = YOLO(model_name)
            train_kwargs = dict(
                data=data_cfg,
                imgsz=training["imgsz"],
                batch=training["batch"],
                epochs=training["epochs"],
                optimizer=training["optimizer"],
                lr0=training["lr0"],
                patience=training["patience"],
                mosaic=training["mosaic"],
                close_mosaic=training["close_mosaic"],
                warmup_epochs=training["warmup_epochs"],
                cos_lr=training["cos_lr"],
                workers=training["workers"],
                seed=seed,
                project=experiment["project_dir"],
                name=exp_name,
                save_period=training.get("save_period", 50),
            )

        # --------------------------
        # ✅ VAL-LOSS MONITOR
        # --------------------------
        model.add_callback(
            "on_fit_epoch_end",
            make_overfit_monitor(window=training.get("overfit_window", 30), exp_name=exp_name),
        )

        # --------------------------
        # TRAIN
        # --------------------------
        start = time.time()

        try:
            results = model.train(**train_kwargs)
        except KeyboardInterrupt:
            print(f"\n⏸️  Training manually interrupted for {exp_name}.")
            print(f"    Last checkpoint should be safe at: {last_ckpt}")
            print("    Re-run this cell — it will auto-resume from that checkpoint.")
            break

        train_time = (time.time() - start) / 60

        # --------------------------
        # TRAIN METRICS
        # --------------------------
        results_csv = Path(results.save_dir) / "results.csv"
        df = pd.read_csv(results_csv)

        map_col = "metrics/mAP50(B)" if "metrics/mAP50(B)" in df.columns else "metrics/mAP50"
        best_row = df.loc[df[map_col].idxmax()]

        best_weights = Path(results.save_dir) / "weights/best.pt"

        # --------------------------
        # VALIDATION
        # --------------------------
        val_results = model.val(data=data_cfg)

        val_map50 = val_results.box.map50
        val_map50_95 = val_results.box.map
        val_precision = val_results.box.mp
        val_recall = val_results.box.mr

        speed = val_results.speed
        total_time = sum(speed.values())
        fps = 1000 / total_time if total_time > 0 else 0

        # --------------------------
        # TEST
        # --------------------------
        test_results = model.val(data=data_cfg, split="test")

        test_map50 = test_results.box.map50
        test_map50_95 = test_results.box.map
        test_precision = test_results.box.mp
        test_recall = test_results.box.mr

        # --------------------------
        # MODEL COMPLEXITY (params + GFLOPs)
        # --------------------------
        model_info = get_model_info(best_weights, imgsz=training["imgsz"])

        # --------------------------
        # STORE
        # --------------------------
        result = {
            "seed": seed,
            "best_epoch": int(best_row["epoch"]),

            "train_mAP50": best_row.get("metrics/mAP50(B)", best_row.get("metrics/mAP50")),
            "train_precision": best_row.get("metrics/precision(B)", best_row.get("metrics/precision")),
            "train_recall": best_row.get("metrics/recall(B)", best_row.get("metrics/recall")),

            "val_mAP50": val_map50,
            "val_mAP50_95": val_map50_95,
            "val_precision": val_precision,
            "val_recall": val_recall,

            "test_mAP50": test_map50,
            "test_mAP50_95": test_map50_95,
            "test_precision": test_precision,
            "test_recall": test_recall,

            "fps": fps,
            "train_time_min": train_time,

            "early_stopping": bool(best_row["epoch"] < training["epochs"]),
            "weights": str(best_weights),
        }

        all_results.append(result)

        print("\n📊 Seed Results:")
        for k, v in result.items():
            if k != "weights":
                print(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}")

        # --------------------------
        # ✅ CSV LOG
        # --------------------------
        log_experiment_to_csv(cfg, seed, exp_name, result, model_info, master_csv_path)
        print(f"📝 Logged to {master_csv_path}")

        del model
        torch.cuda.empty_cache()
        gc.collect()

    # =====================================
    # 📊 FINAL TABLE
    # =====================================
    if not all_results:
        print("\n⚠️  No completed runs to summarize (training interrupted before any seed finished).\n")
        return

    print("\n📊 ALL SEED RESULTS\n")

    df_results = pd.DataFrame(all_results)
    print(df_results.round(4))

    # =====================================
    # 🏆 BEST MODEL
    # =====================================
    best_exp = df_results.loc[df_results["test_mAP50"].idxmax()]

    print("\n🏆 BEST MODEL (TEST mAP50)\n")
    print(best_exp)

    # =====================================
    # 📈 AVERAGE PERFORMANCE
    # =====================================
    print("\n📈 AVERAGE PERFORMANCE\n")
    print(df_results.mean(numeric_only=True).round(4))

    # =====================================
    # 📊 PER-CLASS METRICS
    # =====================================
    print("\n📊 PER-CLASS PERFORMANCE (TEST SET)\n")

    best_model = YOLO(best_exp["weights"])

    test_results = best_model.val(data=data_cfg, split="test")

    names = test_results.names

    precision_cls = test_results.box.p
    recall_cls = test_results.box.r
    map50_cls = test_results.box.ap50

    class_metrics = []

    for i, name in names.items():
        class_metrics.append({
            "class": name,
            "precision": float(precision_cls[i]),
            "recall": float(recall_cls[i]),
            "mAP50": float(map50_cls[i]),
        })

    df_class = pd.DataFrame(class_metrics)
    df_class = df_class.sort_values(by="mAP50", ascending=False)

    print(df_class.round(4))

    print(f"\n✅ DONE — full experiment log at: {master_csv_path}\n")


# =========================================================
# ▶️ RUN
# =========================================================
run()


🚀 Starting Experiments...


========== SEED 1 ==========

⚠️  Found existing checkpoint for NEUDET__YOLO_v11n_SGD__300epochs_seed1 — resuming from D:\MY Projects\Steel Defect Detection\runs\detect\experiments\NEUDET__YOLO_v11n_SGD__300epochs_seed1\weights\last.pt
Ultralytics 8.4.14  Python-3.11.0 torch-2.1.2+cu118 CUDA:0 (NVIDIA GeForce GTX 1650, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=20, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\MY Projects\Steel Defect Detection\configs\data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=300, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False,

In [2]:
# =========================================================
# 📦 IMPORTS + SETUP
# =========================================================
import sys, os, random, time, gc
from pathlib import Path
from datetime import datetime
import yaml
import torch
import pandas as pd

# ---------------------------------------------------------
# 🔥 PROJECT ROOT SETUP
# ---------------------------------------------------------
PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "src").exists() and (p / "configs").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Project root not found")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ---------------------------------------------------------
# 🔥 REGISTER CUSTOM MODULES
# ---------------------------------------------------------
import src
import ultralytics.nn.tasks as _tasks
import ultralytics.nn.modules as _modules

from src.custom_modules import M_C3k2, WeightedConcat, HybridSPDConv_3
from src.spd_conv import SPDConv, SPDHybrid, SPDHybrid_NO_Fuse, DKStem

for name, cls in {
    "M_C3k2": M_C3k2,
    "WeightedConcat": WeightedConcat,
    "HybridSPDConv_3": HybridSPDConv_3,
    "SPDConv": SPDConv,
    "SPDHybrid": SPDHybrid,
    "SPDHybrid_NO_Fuse": SPDHybrid_NO_Fuse,
    "DKStem": DKStem,
}.items():
    _tasks.__dict__[name] = cls
    _modules.__dict__[name] = cls

from ultralytics import YOLO


# =========================================================
# 🔁 SEED CONTROL
# =========================================================
def set_seed(seed):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


# =========================================================
# 📂 LOAD CONFIG
# =========================================================
def load_config():
    cfg_path = PROJECT_ROOT / "configs" / "base.yaml"
    with open(cfg_path, encoding="utf-8") as f:
        cfg = yaml.safe_load(f)

    cfg["model"] = str((PROJECT_ROOT / cfg["model"]).resolve())
    cfg["experiment"]["project_dir"] = str(
        (PROJECT_ROOT / cfg["experiment"]["project_dir"]).resolve()
    )
    cfg["data"]["config"] = str(
        (PROJECT_ROOT / cfg["data"]["config"]).resolve()
    )

    return cfg


# =========================================================
# 🧮 MODEL COMPLEXITY (params + GFLOPs ONLY)
# =========================================================
def get_model_info(weights_path, imgsz=640):
    """
    Loads a checkpoint fresh and calculates total params and GFLOPs directly.
    Returns zero for layers/gradients so downstream table logging doesn't break.
    """
    info_model = YOLO(str(weights_path))
    model = info_model.model

    # 1. Direct PyTorch Parameter Count
    n_p = sum(p.numel() for p in model.parameters())

    # 2. Extract GFLOPs safely
    flops = 0.0
    try:
        info_out = model.info(detailed=False, verbose=False, imgsz=imgsz)
        if isinstance(info_out, (tuple, list)) and len(info_out) >= 4:
            flops = info_out[3]
        elif hasattr(model, "flops"):
            flops = model.flops
    except Exception as e:
        print(f"[warning] Could not calculate GFLOPs: {e}")

    del info_model
    return {"params": n_p, "gflops": flops, "layers": 0, "gradients": 0}


# =========================================================
# 🚨 OVERFITTING / VAL-LOSS-DIVERGENCE MONITOR
# =========================================================
def make_overfit_monitor(window=5, exp_name=""):
    """
    Registers as an `on_fit_epoch_end` callback. Watches for the classic
    overfitting signature: train loss still falling while val loss climbs,
    measured over a rolling `window` of epochs.
    """
    history = {"epoch": [], "train_loss": [], "val_loss": []}

    def _callback(trainer):
        try:
            m = trainer.metrics or {}
            val_loss = sum(
                m.get(k, 0.0) for k in ("val/box_loss", "val/cls_loss", "val/dfl_loss")
            )
            train_loss = (
                float(trainer.tloss.sum())
                if getattr(trainer, "tloss", None) is not None
                else None
            )

            if train_loss is None or val_loss == 0.0:
                return  # metrics not populated yet this epoch

            history["epoch"].append(trainer.epoch)
            history["train_loss"].append(train_loss)
            history["val_loss"].append(val_loss)

            if len(history["epoch"]) >= window:
                t_now, t_prev = history["train_loss"][-1], history["train_loss"][-window]
                v_now, v_prev = history["val_loss"][-1], history["val_loss"][-window]

                if (t_now < t_prev) and (v_now > v_prev):
                    print(
                        f"\n🚨 [{exp_name}] Epoch {trainer.epoch}: "
                        f"train loss ↓ ({t_prev:.4f} → {t_now:.4f}) but "
                        f"val loss ↑ ({v_prev:.4f} → {v_now:.4f}) over last {window} epochs.\n"
                        f"    Looks like overfitting starting — consider Kernel → Interrupt "
                        f"if this keeps repeating.\n"
                    )
        except Exception as e:
            print(f"[monitor warning] could not evaluate loss trend: {e}")

    return _callback


# =========================================================
# 📝 MASTER CSV LOGGER
# =========================================================
def log_experiment_to_csv(cfg, seed, exp_name, result, model_info, csv_path):
    """
    Appends one full row per seed/run to a master CSV.
    """
    training = cfg["training"]

    row = {
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "experiment_name": exp_name,
        "model_config": cfg["model"],
        "dataset_config": cfg["data"]["config"],
        "seed": seed,

        # --- Model complexity ---
        "params_M": round(model_info["params"] / 1e6, 4),
        "gflops": round(model_info["gflops"], 4),
        "layers": model_info["layers"],

        # --- Training setup ---
        "epochs_planned": training["epochs"],
        "best_epoch": result["best_epoch"],
        "early_stopped": result["early_stopping"],
        "optimizer": training["optimizer"],
        "lr0": training["lr0"],
        "batch": training["batch"],
        "imgsz": training["imgsz"],
        "save_period": training.get("save_period", 50),

        # --- Train (best epoch) metrics ---
        "train_mAP50": result["train_mAP50"],
        "train_precision": result["train_precision"],
        "train_recall": result["train_recall"],

        # --- Val metrics ---
        "val_mAP50": result["val_mAP50"],
        "val_mAP50_95": result["val_mAP50_95"],
        "val_precision": result["val_precision"],
        "val_recall": result["val_recall"],

        # --- Test metrics ---
        "test_mAP50": result["test_mAP50"],
        "test_mAP50_95": result["test_mAP50_95"],
        "test_precision": result["test_precision"],
        "test_recall": result["test_recall"],

        # --- Efficiency ---
        "fps": result["fps"],
        "train_time_min": result["train_time_min"],

        "weights_path": result["weights"],
    }

    csv_path = Path(csv_path)
    csv_path.parent.mkdir(parents=True, exist_ok=True)

    df_row = pd.DataFrame([row])
    if csv_path.exists():
        df_row.to_csv(csv_path, mode="a", header=False, index=False)
    else:
        df_row.to_csv(csv_path, mode="w", header=True, index=False)

    return row


# =========================================================
# 🚀 MAIN PIPELINE
# =========================================================
def run():
    cfg = load_config()

    model_name = cfg["model"]
    training = cfg["training"]
    experiment = cfg["experiment"]
    data_cfg = cfg["data"]["config"]

    seeds = experiment["seeds"]

    master_csv_path = experiment.get(
        "log_csv", PROJECT_ROOT / "results" / "experiment_log.csv"
    )

    all_results = []

    print("\n🚀 Starting Experiments...\n")

    # =====================================
    # 🔁 LOOP OVER SEEDS
    # =====================================
    for seed in seeds:

        print(f"\n========== SEED {seed} ==========\n")

        set_seed(seed)

        exp_name = f"{experiment['name']}_seed{seed}"
        exp_dir = Path(experiment["project_dir"]) / exp_name
        last_ckpt = exp_dir / "weights" / "last.pt"

        # --------------------------
        # ✅ RESUME CHECKPOINT
        # --------------------------
        if last_ckpt.exists():
            print(f"⚠️  Found existing checkpoint for {exp_name} — resuming from {last_ckpt}")
            model = YOLO(str(last_ckpt))
            train_kwargs = dict(resume=True)
        else:
            model = YOLO(model_name)
            train_kwargs = dict(
                data=data_cfg,
                imgsz=training["imgsz"],
                batch=training["batch"],
                epochs=training["epochs"],
                optimizer=training["optimizer"],
                lr0=training["lr0"],
                patience=training["patience"],
                mosaic=training["mosaic"],
                close_mosaic=training["close_mosaic"],
                warmup_epochs=training["warmup_epochs"],
                cos_lr=training["cos_lr"],
                workers=training["workers"],
                seed=seed,
                project=experiment["project_dir"],
                name=exp_name,
                save_period=training.get("save_period", 50),
            )

        # --------------------------
        # ✅ VAL-LOSS MONITOR
        # --------------------------
        model.add_callback(
            "on_fit_epoch_end",
            make_overfit_monitor(window=training.get("overfit_window", 30), exp_name=exp_name),
        )

        # --------------------------
        # TRAIN
        # --------------------------
        start = time.time()

        try:
            results = model.train(**train_kwargs)
        except KeyboardInterrupt:
            print(f"\n⏸️  Training manually interrupted for {exp_name}.")
            print(f"    Last checkpoint should be safe at: {last_ckpt}")
            print("    Re-run this cell — it will auto-resume from that checkpoint.")
            break

        train_time = (time.time() - start) / 60

        # --------------------------
        # TRAIN METRICS
        # --------------------------
        results_csv = Path(results.save_dir) / "results.csv"
        df = pd.read_csv(results_csv)

        map_col = "metrics/mAP50(B)" if "metrics/mAP50(B)" in df.columns else "metrics/mAP50"
        best_row = df.loc[df[map_col].idxmax()]

        best_weights = Path(results.save_dir) / "weights/best.pt"

        # --------------------------
        # VALIDATION
        # --------------------------
        val_results = model.val(data=data_cfg)

        val_map50 = val_results.box.map50
        val_map50_95 = val_results.box.map
        val_precision = val_results.box.mp
        val_recall = val_results.box.mr

        speed = val_results.speed
        total_time = sum(speed.values())
        fps = 1000 / total_time if total_time > 0 else 0

        # --------------------------
        # TEST
        # --------------------------
        test_results = model.val(data=data_cfg, split="test")

        test_map50 = test_results.box.map50
        test_map50_95 = test_results.box.map
        test_precision = test_results.box.mp
        test_recall = test_results.box.mr

        # --------------------------
        # MODEL COMPLEXITY (params + GFLOPs)
        # --------------------------
        model_info = get_model_info(best_weights, imgsz=training["imgsz"])

        # --------------------------
        # STORE
        # --------------------------
        result = {
            "seed": seed,
            "best_epoch": int(best_row["epoch"]),

            "train_mAP50": best_row.get("metrics/mAP50(B)", best_row.get("metrics/mAP50")),
            "train_precision": best_row.get("metrics/precision(B)", best_row.get("metrics/precision")),
            "train_recall": best_row.get("metrics/recall(B)", best_row.get("metrics/recall")),

            "val_mAP50": val_map50,
            "val_mAP50_95": val_map50_95,
            "val_precision": val_precision,
            "val_recall": val_recall,

            "test_mAP50": test_map50,
            "test_mAP50_95": test_map50_95,
            "test_precision": test_precision,
            "test_recall": test_recall,

            "fps": fps,
            "train_time_min": train_time,

            "early_stopping": bool(best_row["epoch"] < training["epochs"]),
            "weights": str(best_weights),
        }

        all_results.append(result)

        print("\n📊 Seed Results:")
        for k, v in result.items():
            if k != "weights":
                print(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}")

        # --------------------------
        # ✅ CSV LOG
        # --------------------------
        log_experiment_to_csv(cfg, seed, exp_name, result, model_info, master_csv_path)
        print(f"📝 Logged to {master_csv_path}")

        del model
        torch.cuda.empty_cache()
        gc.collect()

    # =====================================
    # 📊 FINAL TABLE
    # =====================================
    if not all_results:
        print("\n⚠️  No completed runs to summarize (training interrupted before any seed finished).\n")
        return

    print("\n📊 ALL SEED RESULTS\n")

    df_results = pd.DataFrame(all_results)
    print(df_results.round(4))

    # =====================================
    # 🏆 BEST MODEL
    # =====================================
    best_exp = df_results.loc[df_results["test_mAP50"].idxmax()]

    print("\n🏆 BEST MODEL (TEST mAP50)\n")
    print(best_exp)

    # =====================================
    # 📈 AVERAGE PERFORMANCE
    # =====================================
    print("\n📈 AVERAGE PERFORMANCE\n")
    print(df_results.mean(numeric_only=True).round(4))

    # =====================================
    # 📊 PER-CLASS METRICS
    # =====================================
    print("\n📊 PER-CLASS PERFORMANCE (TEST SET)\n")

    best_model = YOLO(best_exp["weights"])

    test_results = best_model.val(data=data_cfg, split="test")

    names = test_results.names

    precision_cls = test_results.box.p
    recall_cls = test_results.box.r
    map50_cls = test_results.box.ap50

    class_metrics = []

    for i, name in names.items():
        class_metrics.append({
            "class": name,
            "precision": float(precision_cls[i]),
            "recall": float(recall_cls[i]),
            "mAP50": float(map50_cls[i]),
        })

    df_class = pd.DataFrame(class_metrics)
    df_class = df_class.sort_values(by="mAP50", ascending=False)

    print(df_class.round(4))

    print(f"\n✅ DONE — full experiment log at: {master_csv_path}\n")


# =========================================================
# ▶️ RUN
# =========================================================
run()


🚀 Starting Experiments...


========== SEED 1 ==========

⚠️  Found existing checkpoint for NEUDET__YOLO_v11n_ADAMW__300epochs_seed1 — resuming from D:\MY Projects\Steel Defect Detection\runs\detect\experiments\NEUDET__YOLO_v11n_ADAMW__300epochs_seed1\weights\last.pt
Ultralytics 8.4.14  Python-3.11.0 torch-2.1.2+cu118 CUDA:0 (NVIDIA GeForce GTX 1650, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=20, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=D:\MY Projects\Steel Defect Detection\configs\data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=300, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=Fal

In [4]:
# =========================================================
# 📦 IMPORTS + SETUP
# =========================================================
import sys, os, random, time, gc
from pathlib import Path
from datetime import datetime
import yaml
import torch
import pandas as pd

# ---------------------------------------------------------
# 🔥 PROJECT ROOT SETUP
# ---------------------------------------------------------
PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "src").exists() and (p / "configs").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Project root not found")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ---------------------------------------------------------
# 🔥 REGISTER CUSTOM MODULES
# ---------------------------------------------------------
import src
import ultralytics.nn.tasks as _tasks
import ultralytics.nn.modules as _modules

from src.custom_modules import M_C3k2, WeightedConcat, HybridSPDConv_3
from src.spd_conv import SPDConv, SPDHybrid, SPDHybrid_NO_Fuse, DKStem

for name, cls in {
    "M_C3k2": M_C3k2,
    "WeightedConcat": WeightedConcat,
    "HybridSPDConv_3": HybridSPDConv_3,
    "SPDConv": SPDConv,
    "SPDHybrid": SPDHybrid,
    "SPDHybrid_NO_Fuse": SPDHybrid_NO_Fuse,
    "DKStem": DKStem,
}.items():
    _tasks.__dict__[name] = cls
    _modules.__dict__[name] = cls

from ultralytics import YOLO


# =========================================================
# 🔁 SEED CONTROL
# =========================================================
def set_seed(seed):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


# =========================================================
# 📂 LOAD CONFIG
# =========================================================
def load_config():
    cfg_path = PROJECT_ROOT / "configs" / "base.yaml"
    with open(cfg_path, encoding="utf-8") as f:
        cfg = yaml.safe_load(f)

    cfg["model"] = str((PROJECT_ROOT / cfg["model"]).resolve())
    cfg["experiment"]["project_dir"] = str(
        (PROJECT_ROOT / cfg["experiment"]["project_dir"]).resolve()
    )
    cfg["data"]["config"] = str(
        (PROJECT_ROOT / cfg["data"]["config"]).resolve()
    )

    return cfg


# =========================================================
# 🧮 MODEL COMPLEXITY (params + GFLOPs ONLY)
# =========================================================
def get_model_info(weights_path, imgsz=640):
    """
    Loads a checkpoint fresh and calculates total params and GFLOPs directly.
    Returns zero for layers/gradients so downstream table logging doesn't break.
    """
    info_model = YOLO(str(weights_path))
    model = info_model.model

    # 1. Direct PyTorch Parameter Count
    n_p = sum(p.numel() for p in model.parameters())

    # 2. Extract GFLOPs safely
    flops = 0.0
    try:
        info_out = model.info(detailed=False, verbose=False, imgsz=imgsz)
        if isinstance(info_out, (tuple, list)) and len(info_out) >= 4:
            flops = info_out[3]
        elif hasattr(model, "flops"):
            flops = model.flops
    except Exception as e:
        print(f"[warning] Could not calculate GFLOPs: {e}")

    del info_model
    return {"params": n_p, "gflops": flops, "layers": 0, "gradients": 0}


# =========================================================
# 🚨 OVERFITTING / VAL-LOSS-DIVERGENCE MONITOR
# =========================================================
def make_overfit_monitor(window=5, exp_name=""):
    """
    Registers as an `on_fit_epoch_end` callback. Watches for the classic
    overfitting signature: train loss still falling while val loss climbs,
    measured over a rolling `window` of epochs.
    """
    history = {"epoch": [], "train_loss": [], "val_loss": []}

    def _callback(trainer):
        try:
            m = trainer.metrics or {}
            val_loss = sum(
                m.get(k, 0.0) for k in ("val/box_loss", "val/cls_loss", "val/dfl_loss")
            )
            train_loss = (
                float(trainer.tloss.sum())
                if getattr(trainer, "tloss", None) is not None
                else None
            )

            if train_loss is None or val_loss == 0.0:
                return  # metrics not populated yet this epoch

            history["epoch"].append(trainer.epoch)
            history["train_loss"].append(train_loss)
            history["val_loss"].append(val_loss)

            if len(history["epoch"]) >= window:
                t_now, t_prev = history["train_loss"][-1], history["train_loss"][-window]
                v_now, v_prev = history["val_loss"][-1], history["val_loss"][-window]

                if (t_now < t_prev) and (v_now > v_prev):
                    print(
                        f"\n🚨 [{exp_name}] Epoch {trainer.epoch}: "
                        f"train loss ↓ ({t_prev:.4f} → {t_now:.4f}) but "
                        f"val loss ↑ ({v_prev:.4f} → {v_now:.4f}) over last {window} epochs.\n"
                        f"    Looks like overfitting starting — consider Kernel → Interrupt "
                        f"if this keeps repeating.\n"
                    )
        except Exception as e:
            print(f"[monitor warning] could not evaluate loss trend: {e}")

    return _callback


# =========================================================
# 📝 MASTER CSV LOGGER
# =========================================================
def log_experiment_to_csv(cfg, seed, exp_name, result, model_info, csv_path):
    """
    Appends one full row per seed/run to a master CSV.
    """
    training = cfg["training"]

    row = {
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "experiment_name": exp_name,
        "model_config": cfg["model"],
        "dataset_config": cfg["data"]["config"],
        "seed": seed,

        # --- Model complexity ---
        "params_M": round(model_info["params"] / 1e6, 4),
        "gflops": round(model_info["gflops"], 4),
        "layers": model_info["layers"],

        # --- Training setup ---
        "epochs_planned": training["epochs"],
        "best_epoch": result["best_epoch"],
        "early_stopped": result["early_stopping"],
        "optimizer": training["optimizer"],
        "lr0": training["lr0"],
        "batch": training["batch"],
        "imgsz": training["imgsz"],
        "save_period": training.get("save_period", 50),

        # --- Train (best epoch) metrics ---
        "train_mAP50": result["train_mAP50"],
        "train_precision": result["train_precision"],
        "train_recall": result["train_recall"],

        # --- Val metrics ---
        "val_mAP50": result["val_mAP50"],
        "val_mAP50_95": result["val_mAP50_95"],
        "val_precision": result["val_precision"],
        "val_recall": result["val_recall"],

        # --- Test metrics ---
        "test_mAP50": result["test_mAP50"],
        "test_mAP50_95": result["test_mAP50_95"],
        "test_precision": result["test_precision"],
        "test_recall": result["test_recall"],

        # --- Efficiency ---
        "fps": result["fps"],
        "train_time_min": result["train_time_min"],

        "weights_path": result["weights"],
    }

    csv_path = Path(csv_path)
    csv_path.parent.mkdir(parents=True, exist_ok=True)

    df_row = pd.DataFrame([row])
    if csv_path.exists():
        df_row.to_csv(csv_path, mode="a", header=False, index=False)
    else:
        df_row.to_csv(csv_path, mode="w", header=True, index=False)

    return row


# =========================================================
# 🚀 MAIN PIPELINE
# =========================================================
def run():
    cfg = load_config()

    model_name = cfg["model"]
    training = cfg["training"]
    experiment = cfg["experiment"]
    data_cfg = cfg["data"]["config"]

    seeds = experiment["seeds"]

    master_csv_path = experiment.get(
        "log_csv", PROJECT_ROOT / "results" / "experiment_log.csv"
    )

    all_results = []

    print("\n🚀 Starting Experiments...\n")

    # =====================================
    # 🔁 LOOP OVER SEEDS
    # =====================================
    for seed in seeds:

        print(f"\n========== SEED {seed} ==========\n")

        set_seed(seed)

        exp_name = f"{experiment['name']}_seed{seed}"
        exp_dir = Path(experiment["project_dir"]) / exp_name
        last_ckpt = exp_dir / "weights" / "last.pt"

        # --------------------------
        # ✅ RESUME CHECKPOINT
        # --------------------------
        if last_ckpt.exists():
            print(f"⚠️  Found existing checkpoint for {exp_name} — resuming from {last_ckpt}")
            model = YOLO(str(last_ckpt))
            train_kwargs = dict(resume=True)
        else:
            model = YOLO(model_name)
            train_kwargs = dict(
                data=data_cfg,
                imgsz=training["imgsz"],
                batch=training["batch"],
                epochs=training["epochs"],
                optimizer=training["optimizer"],
                lr0=training["lr0"],
                patience=training["patience"],
                mosaic=training["mosaic"],
                close_mosaic=training["close_mosaic"],
                warmup_epochs=training["warmup_epochs"],
                cos_lr=training["cos_lr"],
                workers=training["workers"],
                seed=seed,
                project=experiment["project_dir"],
                name=exp_name,
                save_period=training.get("save_period", 50),
            )

        # --------------------------
        # ✅ VAL-LOSS MONITOR
        # --------------------------
        model.add_callback(
            "on_fit_epoch_end",
            make_overfit_monitor(window=training.get("overfit_window", 30), exp_name=exp_name),
        )

        # --------------------------
        # TRAIN
        # --------------------------
        start = time.time()

        try:
            results = model.train(**train_kwargs)
        except KeyboardInterrupt:
            print(f"\n⏸️  Training manually interrupted for {exp_name}.")
            print(f"    Last checkpoint should be safe at: {last_ckpt}")
            print("    Re-run this cell — it will auto-resume from that checkpoint.")
            break

        train_time = (time.time() - start) / 60

        # --------------------------
        # TRAIN METRICS
        # --------------------------
        results_csv = Path(results.save_dir) / "results.csv"
        df = pd.read_csv(results_csv)

        map_col = "metrics/mAP50(B)" if "metrics/mAP50(B)" in df.columns else "metrics/mAP50"
        best_row = df.loc[df[map_col].idxmax()]

        best_weights = Path(results.save_dir) / "weights/best.pt"

        # --------------------------
        # VALIDATION
        # --------------------------
        val_results = model.val(data=data_cfg)

        val_map50 = val_results.box.map50
        val_map50_95 = val_results.box.map
        val_precision = val_results.box.mp
        val_recall = val_results.box.mr

        speed = val_results.speed
        total_time = sum(speed.values())
        fps = 1000 / total_time if total_time > 0 else 0

        # --------------------------
        # TEST
        # --------------------------
        test_results = model.val(data=data_cfg, split="test")

        test_map50 = test_results.box.map50
        test_map50_95 = test_results.box.map
        test_precision = test_results.box.mp
        test_recall = test_results.box.mr

        # --------------------------
        # MODEL COMPLEXITY (params + GFLOPs)
        # --------------------------
        model_info = get_model_info(best_weights, imgsz=training["imgsz"])

        # --------------------------
        # STORE
        # --------------------------
        result = {
            "seed": seed,
            "best_epoch": int(best_row["epoch"]),

            "train_mAP50": best_row.get("metrics/mAP50(B)", best_row.get("metrics/mAP50")),
            "train_precision": best_row.get("metrics/precision(B)", best_row.get("metrics/precision")),
            "train_recall": best_row.get("metrics/recall(B)", best_row.get("metrics/recall")),

            "val_mAP50": val_map50,
            "val_mAP50_95": val_map50_95,
            "val_precision": val_precision,
            "val_recall": val_recall,

            "test_mAP50": test_map50,
            "test_mAP50_95": test_map50_95,
            "test_precision": test_precision,
            "test_recall": test_recall,

            "fps": fps,
            "train_time_min": train_time,

            "early_stopping": bool(best_row["epoch"] < training["epochs"]),
            "weights": str(best_weights),
        }

        all_results.append(result)

        print("\n📊 Seed Results:")
        for k, v in result.items():
            if k != "weights":
                print(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}")

        # --------------------------
        # ✅ CSV LOG
        # --------------------------
        log_experiment_to_csv(cfg, seed, exp_name, result, model_info, master_csv_path)
        print(f"📝 Logged to {master_csv_path}")

        del model
        torch.cuda.empty_cache()
        gc.collect()

    # =====================================
    # 📊 FINAL TABLE
    # =====================================
    if not all_results:
        print("\n⚠️  No completed runs to summarize (training interrupted before any seed finished).\n")
        return

    print("\n📊 ALL SEED RESULTS\n")

    df_results = pd.DataFrame(all_results)
    print(df_results.round(4))

    # =====================================
    # 🏆 BEST MODEL
    # =====================================
    best_exp = df_results.loc[df_results["test_mAP50"].idxmax()]

    print("\n🏆 BEST MODEL (TEST mAP50)\n")
    print(best_exp)

    # =====================================
    # 📈 AVERAGE PERFORMANCE
    # =====================================
    print("\n📈 AVERAGE PERFORMANCE\n")
    print(df_results.mean(numeric_only=True).round(4))

    # =====================================
    # 📊 PER-CLASS METRICS
    # =====================================
    print("\n📊 PER-CLASS PERFORMANCE (TEST SET)\n")

    best_model = YOLO(best_exp["weights"])

    test_results = best_model.val(data=data_cfg, split="test")

    names = test_results.names

    precision_cls = test_results.box.p
    recall_cls = test_results.box.r
    map50_cls = test_results.box.ap50

    class_metrics = []

    for i, name in names.items():
        class_metrics.append({
            "class": name,
            "precision": float(precision_cls[i]),
            "recall": float(recall_cls[i]),
            "mAP50": float(map50_cls[i]),
        })

    df_class = pd.DataFrame(class_metrics)
    df_class = df_class.sort_values(by="mAP50", ascending=False)

    print(df_class.round(4))

    print(f"\n✅ DONE — full experiment log at: {master_csv_path}\n")


# =========================================================
# ▶️ RUN
# =========================================================
run()


🚀 Starting Experiments...


========== SEED 1 ==========

WARNING no model scale passed. Assuming scale='n'.


Ultralytics 8.4.14  Python-3.11.0 torch-2.1.2+cu118 CUDA:0 (NVIDIA GeForce GTX 1650, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=20, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=D:\MY Projects\Steel Defect Detection\configs\data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=300, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=D:\MY Projects\Steel Defect Detection\models\V2_DK_SPDHY_yolo11.yaml, momentum=0.937, mosaic=0, multi_scale=0.0, name=NEUDET_DK_SPDHY_YOLO_

In [3]:
# =========================================================
# 📦 IMPORTS + SETUP
# =========================================================
import sys, os, random, time, gc
from pathlib import Path
from datetime import datetime
import yaml
import torch
import pandas as pd

# ---------------------------------------------------------
# 🔥 PROJECT ROOT SETUP
# ---------------------------------------------------------
PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "src").exists() and (p / "configs").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Project root not found")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ---------------------------------------------------------
# 🔥 REGISTER CUSTOM MODULES
# ---------------------------------------------------------
import src
import ultralytics.nn.tasks as _tasks
import ultralytics.nn.modules as _modules

from src.custom_modules import M_C3k2, WeightedConcat, HybridSPDConv_3
from src.spd_conv import SPDConv, SPDHybrid, SPDHybrid_NO_Fuse, DKStem, SPDHybrid_old, SPDHybrid_NO_Fuse_old

for name, cls in {
    "M_C3k2": M_C3k2,
    "WeightedConcat": WeightedConcat,
    "HybridSPDConv_3": HybridSPDConv_3,
    "SPDConv": SPDConv,
    "SPDHybrid": SPDHybrid,
    "SPDHybrid_NO_Fuse": SPDHybrid_NO_Fuse,
    "SPDHybrid_old": SPDHybrid_old,
    "SPDHybrid_NO_Fuse_old": SPDHybrid_NO_Fuse_old,
    "DKStem": DKStem,
}.items():
    _tasks.__dict__[name] = cls
    _modules.__dict__[name] = cls

from ultralytics import YOLO


# =========================================================
# 🔁 SEED CONTROL
# =========================================================
def set_seed(seed):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


# =========================================================
# 📂 LOAD CONFIG
# =========================================================
def load_config():
    cfg_path = PROJECT_ROOT / "configs" / "base.yaml"
    with open(cfg_path, encoding="utf-8") as f:
        cfg = yaml.safe_load(f)

    cfg["model"] = str((PROJECT_ROOT / cfg["model"]).resolve())
    cfg["experiment"]["project_dir"] = str(
        (PROJECT_ROOT / cfg["experiment"]["project_dir"]).resolve()
    )
    cfg["data"]["config"] = str(
        (PROJECT_ROOT / cfg["data"]["config"]).resolve()
    )

    return cfg


# =========================================================
# 🧮 MODEL COMPLEXITY (params + GFLOPs ONLY)
# =========================================================
def get_model_info(weights_path, imgsz=640):
    """
    Loads a checkpoint fresh and calculates total params and GFLOPs directly.
    Returns zero for layers/gradients so downstream table logging doesn't break.
    """
    info_model = YOLO(str(weights_path))
    model = info_model.model

    # 1. Direct PyTorch Parameter Count
    n_p = sum(p.numel() for p in model.parameters())

    # 2. Extract GFLOPs safely
    flops = 0.0
    try:
        info_out = model.info(detailed=False, verbose=False, imgsz=imgsz)
        if isinstance(info_out, (tuple, list)) and len(info_out) >= 4:
            flops = info_out[3]
        elif hasattr(model, "flops"):
            flops = model.flops
    except Exception as e:
        print(f"[warning] Could not calculate GFLOPs: {e}")

    del info_model
    return {"params": n_p, "gflops": flops, "layers": 0, "gradients": 0}


# =========================================================
# 🚨 OVERFITTING / VAL-LOSS-DIVERGENCE MONITOR
# =========================================================
def make_overfit_monitor(window=5, exp_name=""):
    """
    Registers as an `on_fit_epoch_end` callback. Watches for the classic
    overfitting signature: train loss still falling while val loss climbs,
    measured over a rolling `window` of epochs.
    """
    history = {"epoch": [], "train_loss": [], "val_loss": []}

    def _callback(trainer):
        try:
            m = trainer.metrics or {}
            val_loss = sum(
                m.get(k, 0.0) for k in ("val/box_loss", "val/cls_loss", "val/dfl_loss")
            )
            train_loss = (
                float(trainer.tloss.sum())
                if getattr(trainer, "tloss", None) is not None
                else None
            )

            if train_loss is None or val_loss == 0.0:
                return  # metrics not populated yet this epoch

            history["epoch"].append(trainer.epoch)
            history["train_loss"].append(train_loss)
            history["val_loss"].append(val_loss)

            if len(history["epoch"]) >= window:
                t_now, t_prev = history["train_loss"][-1], history["train_loss"][-window]
                v_now, v_prev = history["val_loss"][-1], history["val_loss"][-window]

                if (t_now < t_prev) and (v_now > v_prev):
                    print(
                        f"\n🚨 [{exp_name}] Epoch {trainer.epoch}: "
                        f"train loss ↓ ({t_prev:.4f} → {t_now:.4f}) but "
                        f"val loss ↑ ({v_prev:.4f} → {v_now:.4f}) over last {window} epochs.\n"
                        f"    Looks like overfitting starting — consider Kernel → Interrupt "
                        f"if this keeps repeating.\n"
                    )
        except Exception as e:
            print(f"[monitor warning] could not evaluate loss trend: {e}")

    return _callback


# =========================================================
# 📝 MASTER CSV LOGGER
# =========================================================
def log_experiment_to_csv(cfg, seed, exp_name, result, model_info, csv_path):
    """
    Appends one full row per seed/run to a master CSV.
    """
    training = cfg["training"]

    row = {
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "experiment_name": exp_name,
        "model_config": cfg["model"],
        "dataset_config": cfg["data"]["config"],
        "seed": seed,

        # --- Model complexity ---
        "params_M": round(model_info["params"] / 1e6, 4),
        "gflops": round(model_info["gflops"], 4),
        "layers": model_info["layers"],

        # --- Training setup ---
        "epochs_planned": training["epochs"],
        "best_epoch": result["best_epoch"],
        "early_stopped": result["early_stopping"],
        "optimizer": training["optimizer"],
        "lr0": training["lr0"],
        "batch": training["batch"],
        "imgsz": training["imgsz"],
        "save_period": training.get("save_period", 50),

        # --- Train (best epoch) metrics ---
        "train_mAP50": result["train_mAP50"],
        "train_precision": result["train_precision"],
        "train_recall": result["train_recall"],

        # --- Val metrics ---
        "val_mAP50": result["val_mAP50"],
        "val_mAP50_95": result["val_mAP50_95"],
        "val_precision": result["val_precision"],
        "val_recall": result["val_recall"],

        # --- Test metrics ---
        "test_mAP50": result["test_mAP50"],
        "test_mAP50_95": result["test_mAP50_95"],
        "test_precision": result["test_precision"],
        "test_recall": result["test_recall"],

        # --- Efficiency ---
        "fps": result["fps"],
        "train_time_min": result["train_time_min"],

        "weights_path": result["weights"],
    }

    csv_path = Path(csv_path)
    csv_path.parent.mkdir(parents=True, exist_ok=True)

    df_row = pd.DataFrame([row])
    if csv_path.exists():
        df_row.to_csv(csv_path, mode="a", header=False, index=False)
    else:
        df_row.to_csv(csv_path, mode="w", header=True, index=False)

    return row


# =========================================================
# 🚀 MAIN PIPELINE
# =========================================================
def run():
    cfg = load_config()

    model_name = cfg["model"]
    training = cfg["training"]
    experiment = cfg["experiment"]
    data_cfg = cfg["data"]["config"]

    seeds = experiment["seeds"]

    master_csv_path = experiment.get(
        "log_csv", PROJECT_ROOT / "results" / "experiment_log.csv"
    )

    all_results = []

    print("\n🚀 Starting Experiments...\n")

    # =====================================
    # 🔁 LOOP OVER SEEDS
    # =====================================
    for seed in seeds:

        print(f"\n========== SEED {seed} ==========\n")

        set_seed(seed)

        exp_name = f"{experiment['name']}_seed{seed}"
        exp_dir = Path(experiment["project_dir"]) / exp_name
        last_ckpt = exp_dir / "weights" / "last.pt"

        # --------------------------
        # ✅ RESUME CHECKPOINT
        # --------------------------
        if last_ckpt.exists():
            print(f"⚠️  Found existing checkpoint for {exp_name} — resuming from {last_ckpt}")
            model = YOLO(str(last_ckpt))
            train_kwargs = dict(resume=True)
        else:
            model = YOLO(model_name)
            train_kwargs = dict(
                data=data_cfg,
                imgsz=training["imgsz"],
                batch=training["batch"],
                epochs=training["epochs"],
                optimizer=training["optimizer"],
                lr0=training["lr0"],
                patience=training["patience"],
                mosaic=training["mosaic"],
                close_mosaic=training["close_mosaic"],
                warmup_epochs=training["warmup_epochs"],
                cos_lr=training["cos_lr"],
                workers=training["workers"],
                seed=seed,
                project=experiment["project_dir"],
                name=exp_name,
                save_period=training.get("save_period", 50),
            )

        # --------------------------
        # ✅ VAL-LOSS MONITOR
        # --------------------------
        model.add_callback(
            "on_fit_epoch_end",
            make_overfit_monitor(window=training.get("overfit_window", 30), exp_name=exp_name),
        )

        # --------------------------
        # TRAIN
        # --------------------------
        start = time.time()

        try:
            results = model.train(**train_kwargs)
        except KeyboardInterrupt:
            print(f"\n⏸️  Training manually interrupted for {exp_name}.")
            print(f"    Last checkpoint should be safe at: {last_ckpt}")
            print("    Re-run this cell — it will auto-resume from that checkpoint.")
            break

        train_time = (time.time() - start) / 60

        # --------------------------
        # TRAIN METRICS
        # --------------------------
        results_csv = Path(results.save_dir) / "results.csv"
        df = pd.read_csv(results_csv)

        map_col = "metrics/mAP50(B)" if "metrics/mAP50(B)" in df.columns else "metrics/mAP50"
        best_row = df.loc[df[map_col].idxmax()]

        best_weights = Path(results.save_dir) / "weights/best.pt"

        # --------------------------
        # VALIDATION
        # --------------------------
        val_results = model.val(data=data_cfg)

        val_map50 = val_results.box.map50
        val_map50_95 = val_results.box.map
        val_precision = val_results.box.mp
        val_recall = val_results.box.mr

        speed = val_results.speed
        total_time = sum(speed.values())
        fps = 1000 / total_time if total_time > 0 else 0

        # --------------------------
        # TEST
        # --------------------------
        test_results = model.val(data=data_cfg, split="test")

        test_map50 = test_results.box.map50
        test_map50_95 = test_results.box.map
        test_precision = test_results.box.mp
        test_recall = test_results.box.mr

        # --------------------------
        # MODEL COMPLEXITY (params + GFLOPs)
        # --------------------------
        model_info = get_model_info(best_weights, imgsz=training["imgsz"])

        # --------------------------
        # STORE
        # --------------------------
        result = {
            "seed": seed,
            "best_epoch": int(best_row["epoch"]),

            "train_mAP50": best_row.get("metrics/mAP50(B)", best_row.get("metrics/mAP50")),
            "train_precision": best_row.get("metrics/precision(B)", best_row.get("metrics/precision")),
            "train_recall": best_row.get("metrics/recall(B)", best_row.get("metrics/recall")),

            "val_mAP50": val_map50,
            "val_mAP50_95": val_map50_95,
            "val_precision": val_precision,
            "val_recall": val_recall,

            "test_mAP50": test_map50,
            "test_mAP50_95": test_map50_95,
            "test_precision": test_precision,
            "test_recall": test_recall,

            "fps": fps,
            "train_time_min": train_time,

            "early_stopping": bool(best_row["epoch"] < training["epochs"]),
            "weights": str(best_weights),
        }

        all_results.append(result)

        print("\n📊 Seed Results:")
        for k, v in result.items():
            if k != "weights":
                print(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}")

        # --------------------------
        # ✅ CSV LOG
        # --------------------------
        log_experiment_to_csv(cfg, seed, exp_name, result, model_info, master_csv_path)
        print(f"📝 Logged to {master_csv_path}")

        del model
        torch.cuda.empty_cache()
        gc.collect()

    # =====================================
    # 📊 FINAL TABLE
    # =====================================
    if not all_results:
        print("\n⚠️  No completed runs to summarize (training interrupted before any seed finished).\n")
        return

    print("\n📊 ALL SEED RESULTS\n")

    df_results = pd.DataFrame(all_results)
    print(df_results.round(4))

    # =====================================
    # 🏆 BEST MODEL
    # =====================================
    best_exp = df_results.loc[df_results["test_mAP50"].idxmax()]

    print("\n🏆 BEST MODEL (TEST mAP50)\n")
    print(best_exp)

    # =====================================
    # 📈 AVERAGE PERFORMANCE
    # =====================================
    print("\n📈 AVERAGE PERFORMANCE\n")
    print(df_results.mean(numeric_only=True).round(4))

    # =====================================
    # 📊 PER-CLASS METRICS
    # =====================================
    print("\n📊 PER-CLASS PERFORMANCE (TEST SET)\n")

    best_model = YOLO(best_exp["weights"])

    test_results = best_model.val(data=data_cfg, split="test")

    names = test_results.names

    precision_cls = test_results.box.p
    recall_cls = test_results.box.r
    map50_cls = test_results.box.ap50

    class_metrics = []

    for i, name in names.items():
        class_metrics.append({
            "class": name,
            "precision": float(precision_cls[i]),
            "recall": float(recall_cls[i]),
            "mAP50": float(map50_cls[i]),
        })

    df_class = pd.DataFrame(class_metrics)
    df_class = df_class.sort_values(by="mAP50", ascending=False)

    print(df_class.round(4))

    print(f"\n✅ DONE — full experiment log at: {master_csv_path}\n")


# =========================================================
# ▶️ RUN
# =========================================================
run()


🚀 Starting Experiments...


========== SEED 1 ==========

WARNING no model scale passed. Assuming scale='n'.


New https://pypi.org/project/ultralytics/8.4.121 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.14  Python-3.11.0 torch-2.1.2+cu118 CUDA:0 (NVIDIA GeForce GTX 1650, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=20, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=D:\MY Projects\Steel Defect Detection\configs\data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=300, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=D:\MY Projects\Steel Defect Detection\

In [1]:
# =========================================================
# 📦 IMPORTS + SETUP
# =========================================================
import sys, os, random, time, gc
from pathlib import Path
from datetime import datetime
import yaml
import torch
import pandas as pd

# ---------------------------------------------------------
# 🔥 PROJECT ROOT SETUP
# ---------------------------------------------------------
PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "src").exists() and (p / "configs").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Project root not found")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ---------------------------------------------------------
# 🔥 REGISTER CUSTOM MODULES
# ---------------------------------------------------------
import src
import ultralytics.nn.tasks as _tasks
import ultralytics.nn.modules as _modules

from src.custom_modules import M_C3k2, WeightedConcat, HybridSPDConv_3
from src.spd_conv import SPDConv, SPDHybrid, SPDHybrid_NO_Fuse, DKStem

for name, cls in {
    "M_C3k2": M_C3k2,
    "WeightedConcat": WeightedConcat,
    "HybridSPDConv_3": HybridSPDConv_3,
    "SPDConv": SPDConv,
    "SPDHybrid": SPDHybrid,
    "SPDHybrid_NO_Fuse": SPDHybrid_NO_Fuse,
    "DKStem": DKStem,
}.items():
    _tasks.__dict__[name] = cls
    _modules.__dict__[name] = cls

from ultralytics import YOLO


# =========================================================
# 🔁 SEED CONTROL
# =========================================================
def set_seed(seed):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


# =========================================================
# 📂 LOAD CONFIG
# =========================================================
def load_config():
    cfg_path = PROJECT_ROOT / "configs" / "base.yaml"
    with open(cfg_path, encoding="utf-8") as f:
        cfg = yaml.safe_load(f)

    cfg["model"] = str((PROJECT_ROOT / cfg["model"]).resolve())
    cfg["experiment"]["project_dir"] = str(
        (PROJECT_ROOT / cfg["experiment"]["project_dir"]).resolve()
    )
    cfg["data"]["config"] = str(
        (PROJECT_ROOT / cfg["data"]["config"]).resolve()
    )

    return cfg


# =========================================================
# 🧮 MODEL COMPLEXITY (params + GFLOPs ONLY)
# =========================================================
def get_model_info(weights_path, imgsz=640):
    """
    Loads a checkpoint fresh and calculates total params and GFLOPs directly.
    Returns zero for layers/gradients so downstream table logging doesn't break.
    """
    info_model = YOLO(str(weights_path))
    model = info_model.model

    # 1. Direct PyTorch Parameter Count
    n_p = sum(p.numel() for p in model.parameters())

    # 2. Extract GFLOPs safely
    flops = 0.0
    try:
        info_out = model.info(detailed=False, verbose=False, imgsz=imgsz)
        if isinstance(info_out, (tuple, list)) and len(info_out) >= 4:
            flops = info_out[3]
        elif hasattr(model, "flops"):
            flops = model.flops
    except Exception as e:
        print(f"[warning] Could not calculate GFLOPs: {e}")

    del info_model
    return {"params": n_p, "gflops": flops, "layers": 0, "gradients": 0}


# =========================================================
# 🚨 OVERFITTING / VAL-LOSS-DIVERGENCE MONITOR
# =========================================================
def make_overfit_monitor(window=5, exp_name=""):
    """
    Registers as an `on_fit_epoch_end` callback. Watches for the classic
    overfitting signature: train loss still falling while val loss climbs,
    measured over a rolling `window` of epochs.
    """
    history = {"epoch": [], "train_loss": [], "val_loss": []}

    def _callback(trainer):
        try:
            m = trainer.metrics or {}
            val_loss = sum(
                m.get(k, 0.0) for k in ("val/box_loss", "val/cls_loss", "val/dfl_loss")
            )
            train_loss = (
                float(trainer.tloss.sum())
                if getattr(trainer, "tloss", None) is not None
                else None
            )

            if train_loss is None or val_loss == 0.0:
                return  # metrics not populated yet this epoch

            history["epoch"].append(trainer.epoch)
            history["train_loss"].append(train_loss)
            history["val_loss"].append(val_loss)

            if len(history["epoch"]) >= window:
                t_now, t_prev = history["train_loss"][-1], history["train_loss"][-window]
                v_now, v_prev = history["val_loss"][-1], history["val_loss"][-window]

                if (t_now < t_prev) and (v_now > v_prev):
                    print(
                        f"\n🚨 [{exp_name}] Epoch {trainer.epoch}: "
                        f"train loss ↓ ({t_prev:.4f} → {t_now:.4f}) but "
                        f"val loss ↑ ({v_prev:.4f} → {v_now:.4f}) over last {window} epochs.\n"
                        f"    Looks like overfitting starting — consider Kernel → Interrupt "
                        f"if this keeps repeating.\n"
                    )
        except Exception as e:
            print(f"[monitor warning] could not evaluate loss trend: {e}")

    return _callback


# =========================================================
# 📝 MASTER CSV LOGGER
# =========================================================
def log_experiment_to_csv(cfg, seed, exp_name, result, model_info, csv_path):
    """
    Appends one full row per seed/run to a master CSV.
    """
    training = cfg["training"]

    row = {
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "experiment_name": exp_name,
        "model_config": cfg["model"],
        "dataset_config": cfg["data"]["config"],
        "seed": seed,

        # --- Model complexity ---
        "params_M": round(model_info["params"] / 1e6, 4),
        "gflops": round(model_info["gflops"], 4),
        "layers": model_info["layers"],

        # --- Training setup ---
        "epochs_planned": training["epochs"],
        "best_epoch": result["best_epoch"],
        "early_stopped": result["early_stopping"],
        "optimizer": training["optimizer"],
        "lr0": training["lr0"],
        "batch": training["batch"],
        "imgsz": training["imgsz"],
        "save_period": training.get("save_period", 50),

        # --- Train (best epoch) metrics ---
        "train_mAP50": result["train_mAP50"],
        "train_precision": result["train_precision"],
        "train_recall": result["train_recall"],

        # --- Val metrics ---
        "val_mAP50": result["val_mAP50"],
        "val_mAP50_95": result["val_mAP50_95"],
        "val_precision": result["val_precision"],
        "val_recall": result["val_recall"],

        # --- Test metrics ---
        "test_mAP50": result["test_mAP50"],
        "test_mAP50_95": result["test_mAP50_95"],
        "test_precision": result["test_precision"],
        "test_recall": result["test_recall"],

        # --- Efficiency ---
        "fps": result["fps"],
        "train_time_min": result["train_time_min"],

        "weights_path": result["weights"],
    }

    csv_path = Path(csv_path)
    csv_path.parent.mkdir(parents=True, exist_ok=True)

    df_row = pd.DataFrame([row])
    if csv_path.exists():
        df_row.to_csv(csv_path, mode="a", header=False, index=False)
    else:
        df_row.to_csv(csv_path, mode="w", header=True, index=False)

    return row


# =========================================================
# 🚀 MAIN PIPELINE
# =========================================================
def run():
    cfg = load_config()

    model_name = cfg["model"]
    training = cfg["training"]
    experiment = cfg["experiment"]
    data_cfg = cfg["data"]["config"]

    seeds = experiment["seeds"]

    master_csv_path = experiment.get(
        "log_csv", PROJECT_ROOT / "results" / "experiment_log.csv"
    )

    all_results = []

    print("\n🚀 Starting Experiments...\n")

    # =====================================
    # 🔁 LOOP OVER SEEDS
    # =====================================
    for seed in seeds:

        print(f"\n========== SEED {seed} ==========\n")

        set_seed(seed)

        exp_name = f"{experiment['name']}_seed{seed}"
        exp_dir = Path(experiment["project_dir"]) / exp_name
        last_ckpt = exp_dir / "weights" / "last.pt"

        # --------------------------
        # ✅ RESUME CHECKPOINT
        # --------------------------
        if last_ckpt.exists():
            print(f"⚠️  Found existing checkpoint for {exp_name} — resuming from {last_ckpt}")
            model = YOLO(str(last_ckpt))
            train_kwargs = dict(resume=True)
        else:
            model = YOLO(model_name)
            train_kwargs = dict(
                data=data_cfg,
                imgsz=training["imgsz"],
                batch=training["batch"],
                epochs=training["epochs"],
                optimizer=training["optimizer"],
                lr0=training["lr0"],
                patience=training["patience"],
                mosaic=training["mosaic"],
                close_mosaic=training["close_mosaic"],
                warmup_epochs=training["warmup_epochs"],
                cos_lr=training["cos_lr"],
                workers=training["workers"],
                seed=seed,
                project=experiment["project_dir"],
                name=exp_name,
                save_period=training.get("save_period", 50),
            )

        # --------------------------
        # ✅ VAL-LOSS MONITOR
        # --------------------------
        model.add_callback(
            "on_fit_epoch_end",
            make_overfit_monitor(window=training.get("overfit_window", 30), exp_name=exp_name),
        )

        # --------------------------
        # TRAIN
        # --------------------------
        start = time.time()

        try:
            results = model.train(**train_kwargs)
        except KeyboardInterrupt:
            print(f"\n⏸️  Training manually interrupted for {exp_name}.")
            print(f"    Last checkpoint should be safe at: {last_ckpt}")
            print("    Re-run this cell — it will auto-resume from that checkpoint.")
            break

        train_time = (time.time() - start) / 60

        # --------------------------
        # TRAIN METRICS
        # --------------------------
        results_csv = Path(results.save_dir) / "results.csv"
        df = pd.read_csv(results_csv)

        map_col = "metrics/mAP50(B)" if "metrics/mAP50(B)" in df.columns else "metrics/mAP50"
        best_row = df.loc[df[map_col].idxmax()]

        best_weights = Path(results.save_dir) / "weights/best.pt"

        # --------------------------
        # VALIDATION
        # --------------------------
        val_results = model.val(data=data_cfg)

        val_map50 = val_results.box.map50
        val_map50_95 = val_results.box.map
        val_precision = val_results.box.mp
        val_recall = val_results.box.mr

        speed = val_results.speed
        total_time = sum(speed.values())
        fps = 1000 / total_time if total_time > 0 else 0

        # --------------------------
        # TEST
        # --------------------------
        test_results = model.val(data=data_cfg, split="test")

        test_map50 = test_results.box.map50
        test_map50_95 = test_results.box.map
        test_precision = test_results.box.mp
        test_recall = test_results.box.mr

        # --------------------------
        # MODEL COMPLEXITY (params + GFLOPs)
        # --------------------------
        model_info = get_model_info(best_weights, imgsz=training["imgsz"])

        # --------------------------
        # STORE
        # --------------------------
        result = {
            "seed": seed,
            "best_epoch": int(best_row["epoch"]),

            "train_mAP50": best_row.get("metrics/mAP50(B)", best_row.get("metrics/mAP50")),
            "train_precision": best_row.get("metrics/precision(B)", best_row.get("metrics/precision")),
            "train_recall": best_row.get("metrics/recall(B)", best_row.get("metrics/recall")),

            "val_mAP50": val_map50,
            "val_mAP50_95": val_map50_95,
            "val_precision": val_precision,
            "val_recall": val_recall,

            "test_mAP50": test_map50,
            "test_mAP50_95": test_map50_95,
            "test_precision": test_precision,
            "test_recall": test_recall,

            "fps": fps,
            "train_time_min": train_time,

            "early_stopping": bool(best_row["epoch"] < training["epochs"]),
            "weights": str(best_weights),
        }

        all_results.append(result)

        print("\n📊 Seed Results:")
        for k, v in result.items():
            if k != "weights":
                print(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}")

        # --------------------------
        # ✅ CSV LOG
        # --------------------------
        log_experiment_to_csv(cfg, seed, exp_name, result, model_info, master_csv_path)
        print(f"📝 Logged to {master_csv_path}")

        del model
        torch.cuda.empty_cache()
        gc.collect()

    # =====================================
    # 📊 FINAL TABLE
    # =====================================
    if not all_results:
        print("\n⚠️  No completed runs to summarize (training interrupted before any seed finished).\n")
        return

    print("\n📊 ALL SEED RESULTS\n")

    df_results = pd.DataFrame(all_results)
    print(df_results.round(4))

    # =====================================
    # 🏆 BEST MODEL
    # =====================================
    best_exp = df_results.loc[df_results["test_mAP50"].idxmax()]

    print("\n🏆 BEST MODEL (TEST mAP50)\n")
    print(best_exp)

    # =====================================
    # 📈 AVERAGE PERFORMANCE
    # =====================================
    print("\n📈 AVERAGE PERFORMANCE\n")
    print(df_results.mean(numeric_only=True).round(4))

    # =====================================
    # 📊 PER-CLASS METRICS
    # =====================================
    print("\n📊 PER-CLASS PERFORMANCE (TEST SET)\n")

    best_model = YOLO(best_exp["weights"])

    test_results = best_model.val(data=data_cfg, split="test")

    names = test_results.names

    precision_cls = test_results.box.p
    recall_cls = test_results.box.r
    map50_cls = test_results.box.ap50

    class_metrics = []

    for i, name in names.items():
        class_metrics.append({
            "class": name,
            "precision": float(precision_cls[i]),
            "recall": float(recall_cls[i]),
            "mAP50": float(map50_cls[i]),
        })

    df_class = pd.DataFrame(class_metrics)
    df_class = df_class.sort_values(by="mAP50", ascending=False)

    print(df_class.round(4))

    print(f"\n✅ DONE — full experiment log at: {master_csv_path}\n")


# =========================================================
# ▶️ RUN
# =========================================================
run()


🚀 Starting Experiments...


========== SEED 1 ==========

⚠️  Found existing checkpoint for NEUDET_v8n+SPD_ADAMW__300epochs_seed1 — resuming from D:\MY Projects\Steel Defect Detection\runs\detect\experiments\NEUDET_v8n+SPD_ADAMW__300epochs_seed1\weights\last.pt
Ultralytics 8.4.14  Python-3.11.0 torch-2.1.2+cu118 CUDA:0 (NVIDIA GeForce GTX 1650, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=20, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=D:\MY Projects\Steel Defect Detection\configs\data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=300, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, io

In [4]:
# =========================================================
# 📦 IMPORTS + SETUP
# =========================================================
import sys, os, random, time, gc
from pathlib import Path
from datetime import datetime
import yaml
import torch
import pandas as pd

# ---------------------------------------------------------
# 🔥 PROJECT ROOT SETUP
# ---------------------------------------------------------
PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "src").exists() and (p / "configs").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Project root not found")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ---------------------------------------------------------
# 🔥 REGISTER CUSTOM MODULES
# ---------------------------------------------------------
import src
import ultralytics.nn.tasks as _tasks
import ultralytics.nn.modules as _modules

from src.custom_modules import M_C3k2, WeightedConcat, HybridSPDConv_3
from src.spd_conv import SPDConv, SPDHybrid, SPDHybrid_NO_Fuse, DKStem, SPDHybrid_old, SPDHybrid_NO_Fuse_old

for name, cls in {
    "M_C3k2": M_C3k2,
    "WeightedConcat": WeightedConcat,
    "HybridSPDConv_3": HybridSPDConv_3,
    "SPDConv": SPDConv,
    "SPDHybrid": SPDHybrid,
    "SPDHybrid_NO_Fuse": SPDHybrid_NO_Fuse,
    "SPDHybrid_old": SPDHybrid_old,
    "SPDHybrid_NO_Fuse_old": SPDHybrid_NO_Fuse_old,
    "DKStem": DKStem,
}.items():
    _tasks.__dict__[name] = cls
    _modules.__dict__[name] = cls

from ultralytics import YOLO


# =========================================================
# 🔁 SEED CONTROL
# =========================================================
def set_seed(seed):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


# =========================================================
# 📂 LOAD CONFIG
# =========================================================
def load_config():
    cfg_path = PROJECT_ROOT / "configs" / "base.yaml"
    with open(cfg_path, encoding="utf-8") as f:
        cfg = yaml.safe_load(f)

    cfg["model"] = str((PROJECT_ROOT / cfg["model"]).resolve())
    cfg["experiment"]["project_dir"] = str(
        (PROJECT_ROOT / cfg["experiment"]["project_dir"]).resolve()
    )
    cfg["data"]["config"] = str(
        (PROJECT_ROOT / cfg["data"]["config"]).resolve()
    )

    return cfg


# =========================================================
# 🧮 MODEL COMPLEXITY (params + GFLOPs ONLY)
# =========================================================
def get_model_info(weights_path, imgsz=640):
    """
    Loads a checkpoint fresh and calculates total params and GFLOPs directly.
    Returns zero for layers/gradients so downstream table logging doesn't break.
    """
    info_model = YOLO(str(weights_path))
    model = info_model.model

    # 1. Direct PyTorch Parameter Count
    n_p = sum(p.numel() for p in model.parameters())

    # 2. Extract GFLOPs safely
    flops = 0.0
    try:
        info_out = model.info(detailed=False, verbose=False, imgsz=imgsz)
        if isinstance(info_out, (tuple, list)) and len(info_out) >= 4:
            flops = info_out[3]
        elif hasattr(model, "flops"):
            flops = model.flops
    except Exception as e:
        print(f"[warning] Could not calculate GFLOPs: {e}")

    del info_model
    return {"params": n_p, "gflops": flops, "layers": 0, "gradients": 0}


# =========================================================
# 🚨 OVERFITTING / VAL-LOSS-DIVERGENCE MONITOR
# =========================================================
def make_overfit_monitor(window=5, exp_name=""):
    """
    Registers as an `on_fit_epoch_end` callback. Watches for the classic
    overfitting signature: train loss still falling while val loss climbs,
    measured over a rolling `window` of epochs.
    """
    history = {"epoch": [], "train_loss": [], "val_loss": []}

    def _callback(trainer):
        try:
            m = trainer.metrics or {}
            val_loss = sum(
                m.get(k, 0.0) for k in ("val/box_loss", "val/cls_loss", "val/dfl_loss")
            )
            train_loss = (
                float(trainer.tloss.sum())
                if getattr(trainer, "tloss", None) is not None
                else None
            )

            if train_loss is None or val_loss == 0.0:
                return  # metrics not populated yet this epoch

            history["epoch"].append(trainer.epoch)
            history["train_loss"].append(train_loss)
            history["val_loss"].append(val_loss)

            if len(history["epoch"]) >= window:
                t_now, t_prev = history["train_loss"][-1], history["train_loss"][-window]
                v_now, v_prev = history["val_loss"][-1], history["val_loss"][-window]

                if (t_now < t_prev) and (v_now > v_prev):
                    print(
                        f"\n🚨 [{exp_name}] Epoch {trainer.epoch}: "
                        f"train loss ↓ ({t_prev:.4f} → {t_now:.4f}) but "
                        f"val loss ↑ ({v_prev:.4f} → {v_now:.4f}) over last {window} epochs.\n"
                        f"    Looks like overfitting starting — consider Kernel → Interrupt "
                        f"if this keeps repeating.\n"
                    )
        except Exception as e:
            print(f"[monitor warning] could not evaluate loss trend: {e}")

    return _callback


# =========================================================
# 📝 MASTER CSV LOGGER
# =========================================================
def log_experiment_to_csv(cfg, seed, exp_name, result, model_info, csv_path):
    """
    Appends one full row per seed/run to a master CSV.
    """
    training = cfg["training"]

    row = {
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "experiment_name": exp_name,
        "model_config": cfg["model"],
        "dataset_config": cfg["data"]["config"],
        "seed": seed,

        # --- Model complexity ---
        "params_M": round(model_info["params"] / 1e6, 4),
        "gflops": round(model_info["gflops"], 4),
        "layers": model_info["layers"],

        # --- Training setup ---
        "epochs_planned": training["epochs"],
        "best_epoch": result["best_epoch"],
        "early_stopped": result["early_stopping"],
        "optimizer": training["optimizer"],
        "lr0": training["lr0"],
        "batch": training["batch"],
        "imgsz": training["imgsz"],
        "save_period": training.get("save_period", 50),

        # --- Train (best epoch) metrics ---
        "train_mAP50": result["train_mAP50"],
        "train_precision": result["train_precision"],
        "train_recall": result["train_recall"],

        # --- Val metrics ---
        "val_mAP50": result["val_mAP50"],
        "val_mAP50_95": result["val_mAP50_95"],
        "val_precision": result["val_precision"],
        "val_recall": result["val_recall"],

        # --- Test metrics ---
        "test_mAP50": result["test_mAP50"],
        "test_mAP50_95": result["test_mAP50_95"],
        "test_precision": result["test_precision"],
        "test_recall": result["test_recall"],

        # --- Efficiency ---
        "fps": result["fps"],
        "train_time_min": result["train_time_min"],

        "weights_path": result["weights"],
    }

    csv_path = Path(csv_path)
    csv_path.parent.mkdir(parents=True, exist_ok=True)

    df_row = pd.DataFrame([row])
    if csv_path.exists():
        df_row.to_csv(csv_path, mode="a", header=False, index=False)
    else:
        df_row.to_csv(csv_path, mode="w", header=True, index=False)

    return row


# =========================================================
# 🚀 MAIN PIPELINE
# =========================================================
def run():
    cfg = load_config()

    model_name = cfg["model"]
    training = cfg["training"]
    experiment = cfg["experiment"]
    data_cfg = cfg["data"]["config"]

    seeds = experiment["seeds"]

    master_csv_path = experiment.get(
        "log_csv", PROJECT_ROOT / "results" / "experiment_log.csv"
    )

    all_results = []

    print("\n🚀 Starting Experiments...\n")

    # =====================================
    # 🔁 LOOP OVER SEEDS
    # =====================================
    for seed in seeds:

        print(f"\n========== SEED {seed} ==========\n")

        set_seed(seed)

        exp_name = f"{experiment['name']}_seed{seed}"
        exp_dir = Path(experiment["project_dir"]) / exp_name
        last_ckpt = exp_dir / "weights" / "last.pt"

        # --------------------------
        # ✅ RESUME CHECKPOINT
        # --------------------------
        if last_ckpt.exists():
            print(f"⚠️  Found existing checkpoint for {exp_name} — resuming from {last_ckpt}")
            model = YOLO(str(last_ckpt))
            train_kwargs = dict(resume=True)
        else:
            model = YOLO(model_name)
            train_kwargs = dict(
                data=data_cfg,
                imgsz=training["imgsz"],
                batch=training["batch"],
                epochs=training["epochs"],
                optimizer=training["optimizer"],
                lr0=training["lr0"],
                patience=training["patience"],
                mosaic=training["mosaic"],
                close_mosaic=training["close_mosaic"],
                warmup_epochs=training["warmup_epochs"],
                cos_lr=training["cos_lr"],
                workers=training["workers"],
                seed=seed,
                project=experiment["project_dir"],
                name=exp_name,
                save_period=training.get("save_period", 50),
            )

        # --------------------------
        # ✅ VAL-LOSS MONITOR
        # --------------------------
        model.add_callback(
            "on_fit_epoch_end",
            make_overfit_monitor(window=training.get("overfit_window", 30), exp_name=exp_name),
        )

        # --------------------------
        # TRAIN
        # --------------------------
        start = time.time()

        try:
            results = model.train(**train_kwargs)
        except KeyboardInterrupt:
            print(f"\n⏸️  Training manually interrupted for {exp_name}.")
            print(f"    Last checkpoint should be safe at: {last_ckpt}")
            print("    Re-run this cell — it will auto-resume from that checkpoint.")
            break

        train_time = (time.time() - start) / 60

        # --------------------------
        # TRAIN METRICS
        # --------------------------
        results_csv = Path(results.save_dir) / "results.csv"
        df = pd.read_csv(results_csv)

        map_col = "metrics/mAP50(B)" if "metrics/mAP50(B)" in df.columns else "metrics/mAP50"
        best_row = df.loc[df[map_col].idxmax()]

        best_weights = Path(results.save_dir) / "weights/best.pt"

        # --------------------------
        # VALIDATION
        # --------------------------
        val_results = model.val(data=data_cfg)

        val_map50 = val_results.box.map50
        val_map50_95 = val_results.box.map
        val_precision = val_results.box.mp
        val_recall = val_results.box.mr

        speed = val_results.speed
        total_time = sum(speed.values())
        fps = 1000 / total_time if total_time > 0 else 0

        # --------------------------
        # TEST
        # --------------------------
        test_results = model.val(data=data_cfg, split="test")

        test_map50 = test_results.box.map50
        test_map50_95 = test_results.box.map
        test_precision = test_results.box.mp
        test_recall = test_results.box.mr

        # --------------------------
        # MODEL COMPLEXITY (params + GFLOPs)
        # --------------------------
        model_info = get_model_info(best_weights, imgsz=training["imgsz"])

        # --------------------------
        # STORE
        # --------------------------
        result = {
            "seed": seed,
            "best_epoch": int(best_row["epoch"]),

            "train_mAP50": best_row.get("metrics/mAP50(B)", best_row.get("metrics/mAP50")),
            "train_precision": best_row.get("metrics/precision(B)", best_row.get("metrics/precision")),
            "train_recall": best_row.get("metrics/recall(B)", best_row.get("metrics/recall")),

            "val_mAP50": val_map50,
            "val_mAP50_95": val_map50_95,
            "val_precision": val_precision,
            "val_recall": val_recall,

            "test_mAP50": test_map50,
            "test_mAP50_95": test_map50_95,
            "test_precision": test_precision,
            "test_recall": test_recall,

            "fps": fps,
            "train_time_min": train_time,

            "early_stopping": bool(best_row["epoch"] < training["epochs"]),
            "weights": str(best_weights),
        }

        all_results.append(result)

        print("\n📊 Seed Results:")
        for k, v in result.items():
            if k != "weights":
                print(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}")

        # --------------------------
        # ✅ CSV LOG
        # --------------------------
        log_experiment_to_csv(cfg, seed, exp_name, result, model_info, master_csv_path)
        print(f"📝 Logged to {master_csv_path}")

        del model
        torch.cuda.empty_cache()
        gc.collect()

    # =====================================
    # 📊 FINAL TABLE
    # =====================================
    if not all_results:
        print("\n⚠️  No completed runs to summarize (training interrupted before any seed finished).\n")
        return

    print("\n📊 ALL SEED RESULTS\n")

    df_results = pd.DataFrame(all_results)
    print(df_results.round(4))

    # =====================================
    # 🏆 BEST MODEL
    # =====================================
    best_exp = df_results.loc[df_results["test_mAP50"].idxmax()]

    print("\n🏆 BEST MODEL (TEST mAP50)\n")
    print(best_exp)

    # =====================================
    # 📈 AVERAGE PERFORMANCE
    # =====================================
    print("\n📈 AVERAGE PERFORMANCE\n")
    print(df_results.mean(numeric_only=True).round(4))

    # =====================================
    # 📊 PER-CLASS METRICS
    # =====================================
    print("\n📊 PER-CLASS PERFORMANCE (TEST SET)\n")

    best_model = YOLO(best_exp["weights"])

    test_results = best_model.val(data=data_cfg, split="test")

    names = test_results.names

    precision_cls = test_results.box.p
    recall_cls = test_results.box.r
    map50_cls = test_results.box.ap50

    class_metrics = []

    for i, name in names.items():
        class_metrics.append({
            "class": name,
            "precision": float(precision_cls[i]),
            "recall": float(recall_cls[i]),
            "mAP50": float(map50_cls[i]),
        })

    df_class = pd.DataFrame(class_metrics)
    df_class = df_class.sort_values(by="mAP50", ascending=False)

    print(df_class.round(4))

    print(f"\n✅ DONE — full experiment log at: {master_csv_path}\n")


# =========================================================
# ▶️ RUN
# =========================================================
run()


🚀 Starting Experiments...


========== SEED 1 ==========

WARNING no model scale passed. Assuming scale='n'.
New https://pypi.org/project/ultralytics/8.4.121 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.14  Python-3.11.0 torch-2.1.2+cu118 CUDA:0 (NVIDIA GeForce GTX 1650, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=20, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=D:\MY Projects\Steel Defect Detection\configs\data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=300, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0

In [1]:
# =========================================================
# 📦 IMPORTS + SETUP
# =========================================================
import sys, os, random, time, gc
from pathlib import Path
from datetime import datetime
import yaml
import torch
import pandas as pd

# ---------------------------------------------------------
# 🔥 PROJECT ROOT SETUP
# ---------------------------------------------------------
PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "src").exists() and (p / "configs").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Project root not found")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ---------------------------------------------------------
# 🔥 REGISTER CUSTOM MODULES
# ---------------------------------------------------------
import src
import ultralytics.nn.tasks as _tasks
import ultralytics.nn.modules as _modules

from src.custom_modules import M_C3k2, WeightedConcat, HybridSPDConv_3
from src.spd_conv import SPDConv, SPDHybrid, SPDHybrid_NO_Fuse, DKStem, SPDHybrid_old, SPDHybrid_NO_Fuse_old

for name, cls in {
    "M_C3k2": M_C3k2,
    "WeightedConcat": WeightedConcat,
    "HybridSPDConv_3": HybridSPDConv_3,
    "SPDConv": SPDConv,
    "SPDHybrid": SPDHybrid,
    "SPDHybrid_NO_Fuse": SPDHybrid_NO_Fuse,
    "SPDHybrid_old": SPDHybrid_old,
    "SPDHybrid_NO_Fuse_old": SPDHybrid_NO_Fuse_old,
    "DKStem": DKStem,
}.items():
    _tasks.__dict__[name] = cls
    _modules.__dict__[name] = cls

from ultralytics import YOLO


# =========================================================
# 🔁 SEED CONTROL
# =========================================================
def set_seed(seed):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


# =========================================================
# 📂 LOAD CONFIG
# =========================================================
def load_config():
    cfg_path = PROJECT_ROOT / "configs" / "base.yaml"
    with open(cfg_path, encoding="utf-8") as f:
        cfg = yaml.safe_load(f)

    cfg["model"] = str((PROJECT_ROOT / cfg["model"]).resolve())
    cfg["experiment"]["project_dir"] = str(
        (PROJECT_ROOT / cfg["experiment"]["project_dir"]).resolve()
    )
    cfg["data"]["config"] = str(
        (PROJECT_ROOT / cfg["data"]["config"]).resolve()
    )

    return cfg


# =========================================================
# 🧮 MODEL COMPLEXITY (params + GFLOPs ONLY)
# =========================================================
def get_model_info(weights_path, imgsz=640):
    """
    Loads a checkpoint fresh and calculates total params and GFLOPs directly.
    Returns zero for layers/gradients so downstream table logging doesn't break.
    """
    info_model = YOLO(str(weights_path))
    model = info_model.model

    # 1. Direct PyTorch Parameter Count
    n_p = sum(p.numel() for p in model.parameters())

    # 2. Extract GFLOPs safely
    flops = 0.0
    try:
        info_out = model.info(detailed=False, verbose=False, imgsz=imgsz)
        if isinstance(info_out, (tuple, list)) and len(info_out) >= 4:
            flops = info_out[3]
        elif hasattr(model, "flops"):
            flops = model.flops
    except Exception as e:
        print(f"[warning] Could not calculate GFLOPs: {e}")

    del info_model
    return {"params": n_p, "gflops": flops, "layers": 0, "gradients": 0}


# =========================================================
# 🚨 OVERFITTING / VAL-LOSS-DIVERGENCE MONITOR
# =========================================================
def make_overfit_monitor(window=5, exp_name=""):
    """
    Registers as an `on_fit_epoch_end` callback. Watches for the classic
    overfitting signature: train loss still falling while val loss climbs,
    measured over a rolling `window` of epochs.
    """
    history = {"epoch": [], "train_loss": [], "val_loss": []}

    def _callback(trainer):
        try:
            m = trainer.metrics or {}
            val_loss = sum(
                m.get(k, 0.0) for k in ("val/box_loss", "val/cls_loss", "val/dfl_loss")
            )
            train_loss = (
                float(trainer.tloss.sum())
                if getattr(trainer, "tloss", None) is not None
                else None
            )

            if train_loss is None or val_loss == 0.0:
                return  # metrics not populated yet this epoch

            history["epoch"].append(trainer.epoch)
            history["train_loss"].append(train_loss)
            history["val_loss"].append(val_loss)

            if len(history["epoch"]) >= window:
                t_now, t_prev = history["train_loss"][-1], history["train_loss"][-window]
                v_now, v_prev = history["val_loss"][-1], history["val_loss"][-window]

                if (t_now < t_prev) and (v_now > v_prev):
                    print(
                        f"\n🚨 [{exp_name}] Epoch {trainer.epoch}: "
                        f"train loss ↓ ({t_prev:.4f} → {t_now:.4f}) but "
                        f"val loss ↑ ({v_prev:.4f} → {v_now:.4f}) over last {window} epochs.\n"
                        f"    Looks like overfitting starting — consider Kernel → Interrupt "
                        f"if this keeps repeating.\n"
                    )
        except Exception as e:
            print(f"[monitor warning] could not evaluate loss trend: {e}")

    return _callback


# =========================================================
# 📝 MASTER CSV LOGGER
# =========================================================
def log_experiment_to_csv(cfg, seed, exp_name, result, model_info, csv_path):
    """
    Appends one full row per seed/run to a master CSV.
    """
    training = cfg["training"]

    row = {
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "experiment_name": exp_name,
        "model_config": cfg["model"],
        "dataset_config": cfg["data"]["config"],
        "seed": seed,

        # --- Model complexity ---
        "params_M": round(model_info["params"] / 1e6, 4),
        "gflops": round(model_info["gflops"], 4),
        "layers": model_info["layers"],

        # --- Training setup ---
        "epochs_planned": training["epochs"],
        "best_epoch": result["best_epoch"],
        "early_stopped": result["early_stopping"],
        "optimizer": training["optimizer"],
        "lr0": training["lr0"],
        "batch": training["batch"],
        "imgsz": training["imgsz"],
        "save_period": training.get("save_period", 50),

        # --- Train (best epoch) metrics ---
        "train_mAP50": result["train_mAP50"],
        "train_precision": result["train_precision"],
        "train_recall": result["train_recall"],

        # --- Val metrics ---
        "val_mAP50": result["val_mAP50"],
        "val_mAP50_95": result["val_mAP50_95"],
        "val_precision": result["val_precision"],
        "val_recall": result["val_recall"],

        # --- Test metrics ---
        "test_mAP50": result["test_mAP50"],
        "test_mAP50_95": result["test_mAP50_95"],
        "test_precision": result["test_precision"],
        "test_recall": result["test_recall"],

        # --- Efficiency ---
        "fps": result["fps"],
        "train_time_min": result["train_time_min"],

        "weights_path": result["weights"],
    }

    csv_path = Path(csv_path)
    csv_path.parent.mkdir(parents=True, exist_ok=True)

    df_row = pd.DataFrame([row])
    if csv_path.exists():
        df_row.to_csv(csv_path, mode="a", header=False, index=False)
    else:
        df_row.to_csv(csv_path, mode="w", header=True, index=False)

    return row


# =========================================================
# 🚀 MAIN PIPELINE
# =========================================================
def run():
    cfg = load_config()

    model_name = cfg["model"]
    training = cfg["training"]
    experiment = cfg["experiment"]
    data_cfg = cfg["data"]["config"]

    seeds = experiment["seeds"]

    master_csv_path = experiment.get(
        "log_csv", PROJECT_ROOT / "results" / "experiment_log.csv"
    )

    all_results = []

    print("\n🚀 Starting Experiments...\n")

    # =====================================
    # 🔁 LOOP OVER SEEDS
    # =====================================
    for seed in seeds:

        print(f"\n========== SEED {seed} ==========\n")

        set_seed(seed)

        exp_name = f"{experiment['name']}_seed{seed}"
        exp_dir = Path(experiment["project_dir"]) / exp_name
        last_ckpt = exp_dir / "weights" / "last.pt"

        # --------------------------
        # ✅ RESUME CHECKPOINT
        # --------------------------
        if last_ckpt.exists():
            print(f"⚠️  Found existing checkpoint for {exp_name} — resuming from {last_ckpt}")
            model = YOLO(str(last_ckpt))
            train_kwargs = dict(resume=True)
        else:
            model = YOLO(model_name)
            train_kwargs = dict(
                data=data_cfg,
                imgsz=training["imgsz"],
                batch=training["batch"],
                epochs=training["epochs"],
                optimizer=training["optimizer"],
                lr0=training["lr0"],
                patience=training["patience"],
                mosaic=training["mosaic"],
                close_mosaic=training["close_mosaic"],
                warmup_epochs=training["warmup_epochs"],
                cos_lr=training["cos_lr"],
                workers=training["workers"],
                seed=seed,
                project=experiment["project_dir"],
                name=exp_name,
                save_period=training.get("save_period", 50),
            )

        # --------------------------
        # ✅ VAL-LOSS MONITOR
        # --------------------------
        model.add_callback(
            "on_fit_epoch_end",
            make_overfit_monitor(window=training.get("overfit_window", 30), exp_name=exp_name),
        )

        # --------------------------
        # TRAIN
        # --------------------------
        start = time.time()

        try:
            results = model.train(**train_kwargs)
        except KeyboardInterrupt:
            print(f"\n⏸️  Training manually interrupted for {exp_name}.")
            print(f"    Last checkpoint should be safe at: {last_ckpt}")
            print("    Re-run this cell — it will auto-resume from that checkpoint.")
            break

        train_time = (time.time() - start) / 60

        # --------------------------
        # TRAIN METRICS
        # --------------------------
        results_csv = Path(results.save_dir) / "results.csv"
        df = pd.read_csv(results_csv)

        map_col = "metrics/mAP50(B)" if "metrics/mAP50(B)" in df.columns else "metrics/mAP50"
        best_row = df.loc[df[map_col].idxmax()]

        best_weights = Path(results.save_dir) / "weights/best.pt"

        # --------------------------
        # VALIDATION
        # --------------------------
        val_results = model.val(data=data_cfg)

        val_map50 = val_results.box.map50
        val_map50_95 = val_results.box.map
        val_precision = val_results.box.mp
        val_recall = val_results.box.mr

        speed = val_results.speed
        total_time = sum(speed.values())
        fps = 1000 / total_time if total_time > 0 else 0

        # --------------------------
        # TEST
        # --------------------------
        test_results = model.val(data=data_cfg, split="test")

        test_map50 = test_results.box.map50
        test_map50_95 = test_results.box.map
        test_precision = test_results.box.mp
        test_recall = test_results.box.mr

        # --------------------------
        # MODEL COMPLEXITY (params + GFLOPs)
        # --------------------------
        model_info = get_model_info(best_weights, imgsz=training["imgsz"])

        # --------------------------
        # STORE
        # --------------------------
        result = {
            "seed": seed,
            "best_epoch": int(best_row["epoch"]),

            "train_mAP50": best_row.get("metrics/mAP50(B)", best_row.get("metrics/mAP50")),
            "train_precision": best_row.get("metrics/precision(B)", best_row.get("metrics/precision")),
            "train_recall": best_row.get("metrics/recall(B)", best_row.get("metrics/recall")),

            "val_mAP50": val_map50,
            "val_mAP50_95": val_map50_95,
            "val_precision": val_precision,
            "val_recall": val_recall,

            "test_mAP50": test_map50,
            "test_mAP50_95": test_map50_95,
            "test_precision": test_precision,
            "test_recall": test_recall,

            "fps": fps,
            "train_time_min": train_time,

            "early_stopping": bool(best_row["epoch"] < training["epochs"]),
            "weights": str(best_weights),
        }

        all_results.append(result)

        print("\n📊 Seed Results:")
        for k, v in result.items():
            if k != "weights":
                print(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}")

        # --------------------------
        # ✅ CSV LOG
        # --------------------------
        log_experiment_to_csv(cfg, seed, exp_name, result, model_info, master_csv_path)
        print(f"📝 Logged to {master_csv_path}")

        del model
        torch.cuda.empty_cache()
        gc.collect()

    # =====================================
    # 📊 FINAL TABLE
    # =====================================
    if not all_results:
        print("\n⚠️  No completed runs to summarize (training interrupted before any seed finished).\n")
        return

    print("\n📊 ALL SEED RESULTS\n")

    df_results = pd.DataFrame(all_results)
    print(df_results.round(4))

    # =====================================
    # 🏆 BEST MODEL
    # =====================================
    best_exp = df_results.loc[df_results["test_mAP50"].idxmax()]

    print("\n🏆 BEST MODEL (TEST mAP50)\n")
    print(best_exp)

    # =====================================
    # 📈 AVERAGE PERFORMANCE
    # =====================================
    print("\n📈 AVERAGE PERFORMANCE\n")
    print(df_results.mean(numeric_only=True).round(4))

    # =====================================
    # 📊 PER-CLASS METRICS
    # =====================================
    print("\n📊 PER-CLASS PERFORMANCE (TEST SET)\n")

    best_model = YOLO(best_exp["weights"])

    test_results = best_model.val(data=data_cfg, split="test")

    names = test_results.names

    precision_cls = test_results.box.p
    recall_cls = test_results.box.r
    map50_cls = test_results.box.ap50

    class_metrics = []

    for i, name in names.items():
        class_metrics.append({
            "class": name,
            "precision": float(precision_cls[i]),
            "recall": float(recall_cls[i]),
            "mAP50": float(map50_cls[i]),
        })

    df_class = pd.DataFrame(class_metrics)
    df_class = df_class.sort_values(by="mAP50", ascending=False)

    print(df_class.round(4))

    print(f"\n✅ DONE — full experiment log at: {master_csv_path}\n")


# =========================================================
# ▶️ RUN
# =========================================================
run()


🚀 Starting Experiments...


========== SEED 1 ==========

WARNING no model scale passed. Assuming scale='n'.
Ultralytics 8.4.14  Python-3.11.0 torch-2.1.2+cu118 CUDA:0 (NVIDIA GeForce GTX 1650, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=20, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=D:\MY Projects\Steel Defect Detection\configs\data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=300, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=D:\MY Projects\Steel Defect D